# FI-939: where a strong ARC-AGI-3 agent loses its points - reproduction notebook

Reproduces the paper's power analysis (Figure 2), the probe audit, the within-game control-family table (T4) and the pre-registered targeted gate from the commit-run traces. Every analysis module is embedded verbatim from the source repository at build time.

In [ ]:
# embedded verbatim from fi939-agent-rebuild/duck_score_estimate.py
_SRC = '#!/usr/bin/env python3\n"""FI-939: estimate the OFFICIAL ARC-AGI-3 score of a recorded run (offline, stdlib only).\n\nMethodology (docs.arcprize.org/methodology.md, read 2026-09-26):\n  level_score = min((human_baseline_actions / ai_actions) ** 2, LEVEL_CAP)   for each COMPLETED level\n  game_score  = sum(L * level_score_L over completed L) / sum(L over all levels)   (L = 1-based level number)\n  total       = mean game_score over games (unplayed / uncompleted levels contribute 0)\nBaselines: baseline_actions in each game\'s environment_files/<game>/<ver>/metadata.json.\n\nThe docs do not say how failed attempts count, but the arc_agi package\'s own scorer (duck-venv\narc_agi/scorecard.py, read 2026-09-26) does: level actions = running action total at the level-up minus the\ntotal at the previous level-up, and every RESET adds 1 (inc_reset_count). So:\n  cum   every action taken while on the level, all attempts, RESET included  = the OFFICIAL count\n  last  actions of the final, successful attempt only                         (what a perfect-retry agent would get)\nPer-level score is capped at 1.15 and the game score at the weight share of completed levels (min(score,\nmax_score) in EnvironmentScoreCalculator.to_score), both mirrored here. Assumes the Kaggle gateway runs the\nsame scorer version as the local package.\n\nAlso reported: `compl` = the game score if every completed level scored 1.0, which separates what is lost to\naction inefficiency from what is lost to levels never completed.\nUsage: duck_score_estimate.py <run_dir> [game_substr ...] [--env-dir DIR] [--json out.json] [--levels]\n"""\nimport glob, json, os, sys\n\nLEVEL_CAP = 1.15\n# competition environment_files (metadata.json per game); override with FI939_ENV_DIR\nENV_DIR = os.environ.get("FI939_ENV_DIR", "/mnt/c/arc-agi-3/arc-agi-3-benchmarking/environment_files")\n\n\nHIGH_VARIANCE_GAMES = ("ft09",)\n\n\ndef load_baselines(env_dir):\n    """{game_id (\'ft09-0d8bbf25\') and short id (\'ft09\'): [baseline actions per level]}."""\n    out = {}\n    for p in glob.glob(os.path.join(env_dir, "*", "*", "metadata.json")):\n        try:\n            m = json.load(open(p, encoding="utf-8-sig"))\n        except Exception:\n            continue\n        b = m.get("baseline_actions")\n        gid = m.get("game_id")\n        if not gid or not isinstance(b, list) or not b:\n            continue\n        out[gid] = [int(x) for x in b]\n        out.setdefault(gid.split("-")[0], [int(x) for x in b])\n    return out\n\n\ndef level_score(base, ai, cap=LEVEL_CAP):\n    if not ai or ai <= 0 or not base:\n        return 0.0\n    return min((float(base) / float(ai)) ** 2, cap)\n\n\ndef game_score(baselines, completed):\n    """completed: {level_number(1-based): level_score}. Weighted by level number over ALL levels."""\n    n = len(baselines)\n    den = n * (n + 1) / 2.0\n    if not den:\n        return 0.0\n    raw = sum(L * s for L, s in completed.items() if 1 <= L <= n) / den\n    cap = sum(L for L, s in completed.items() if 1 <= L <= n and s > 0) / den   # scorer: min(score, max_score)\n    return min(raw, cap)\n\n\ndef level_actions(events):\n    """Walk action events. Returns [(level_number, cum_actions, last_attempt_actions)] per completed level."""\n    done, score, cum, last = [], 0, 0, 0\n    for e in events:\n        if e.get("type") != "action":\n            continue\n        cum += 1\n        if str(e.get("action_name") or "").upper() == "RESET":\n            last = 0          # a RESET starts a new attempt; it is not part of the successful attempt\n        else:\n            last += 1\n        sc = e.get("score")\n        if isinstance(sc, (int, float)) and int(sc) > score:\n            for lv in range(score + 1, int(sc) + 1):   # normally one level at a time\n                done.append((lv, cum, last))\n                cum = last = 0\n            score = int(sc)\n        elif e.get("game_over"):\n            last = 0          # the attempt died; the next attempt starts fresh (the RESET, if any, is not counted in last)\n    return done\n\n\ndef analyze(path, baselines):\n    game = os.path.basename(path).split("_p0_events")[0]\n    base = baselines.get(game) or baselines.get(game.split("-")[0])\n    events = []\n    with open(path) as f:\n        for line in f:\n            try:\n                events.append(json.loads(line))\n            except Exception:\n                pass\n    lv = level_actions(events)\n    n_act = sum(1 for e in events if e.get("type") == "action")\n    if not base:\n        return {"game": game, "baseline": None, "levels": len(lv), "actions": n_act}\n    cum = {L: level_score(base[L - 1], c) for L, c, _ in lv if L <= len(base)}\n    last = {L: level_score(base[L - 1], a) for L, _, a in lv if L <= len(base)}\n    full = {L: 1.0 for L, _, _ in lv if L <= len(base)}\n    return {"game": game, "baseline": base, "n_levels": len(base), "levels": len(lv), "actions": n_act,\n            "per_level": [{"level": L, "base": base[L - 1] if L <= len(base) else None, "cum": c, "last": a,\n                           "score_cum": round(cum.get(L, 0.0), 4), "score_last": round(last.get(L, 0.0), 4)}\n                          for L, c, a in lv],\n            "game_cum": game_score(base, cum), "game_last": game_score(base, last), "game_compl": game_score(base, full)}\n\n\ndef max_influence(scored, key="game_cum"):\n    """(game, share of the summed score in %, mean without that game in %) for the game whose removal changes the\n    mean most. Ties go to the larger score."""\n    tot = sum(r[key] for r in scored)\n    n = len(scored)\n    mean = tot / n\n    best = max(scored, key=lambda r: (abs((tot - r[key]) / (n - 1) - mean), r[key]))\n    share = 100.0 * best[key] / tot if tot else 0.0\n    return best["game"].split("-")[0], share, 100.0 * (tot - best[key]) / (n - 1)\n\n\ndef dedupe_by_game(paths):\n    """{basename: path}, keeping the shallowest path for each basename (ties: lexical order)."""\n    out = {}\n    for pth in sorted(paths, key=lambda x: (x.count(os.sep), x)):\n        out.setdefault(os.path.basename(pth), pth)\n    return dict(sorted(out.items()))\n\n\ndef main(argv):\n    env_dir, out_json, show_levels, args = ENV_DIR, None, False, []\n    i = 0\n    while i < len(argv):\n        if argv[i] == "--env-dir":\n            env_dir = argv[i + 1]; i += 2\n        elif argv[i] == "--json":\n            out_json = argv[i + 1]; i += 2\n        elif argv[i] == "--levels":\n            show_levels = True; i += 1\n        else:\n            args.append(argv[i]); i += 1\n    if not args:\n        print(__doc__); return 2\n    bl = load_baselines(env_dir)\n    if not bl:\n        print("NO_BASELINES: no metadata.json with baseline_actions under %s" % env_dir); return 3\n    subs = args[1:]\n    paths = sorted(glob.glob(os.path.join(args[0], "**", "*_p0_events.jsonl"), recursive=True))\n    # FI-939 2026-10-08: M2 trace pulls hold the same game at the top level AND under artifacts/ (byte-identical),\n    # so the recursive glob counted 6 games twice (31 "games" for a 25-game run). One path per game: shallowest.\n    paths = list(dedupe_by_game(paths).values())\n    res = [analyze(p, bl) for p in paths if not subs or any(x in os.path.basename(p) for x in subs)]\n    scored = [r for r in res if r.get("baseline")]\n    print("%-15s %6s %7s %8s %9s %9s %9s" % ("game", "levels", "actions", "n_lvls", "score_cum", "score_last", "compl"))\n    for r in res:\n        if not r.get("baseline"):\n            print("%-15s %6d %7d  (no baseline; excluded)" % (r["game"][:15], r["levels"], r["actions"])); continue\n        print("%-15s %6d %7d %8d %8.2f%% %8.2f%% %8.2f%%" % (r["game"][:15], r["levels"], r["actions"], r["n_levels"],\n              100 * r["game_cum"], 100 * r["game_last"], 100 * r["game_compl"]))\n        if show_levels:\n            for d in r["per_level"]:\n                print("      L%-2d base=%-4s ai_cum=%-4d ai_last=%-4d score_cum=%.4f score_last=%.4f" % (\n                    d["level"], d["base"], d["cum"], d["last"], d["score_cum"], d["score_last"]))\n    if scored:\n        m = lambda k: 100.0 * sum(r[k] for r in scored) / len(scored)\n        # FI-939 2026-09-27: ft09 alone swings the small-sample mean ~2x (sonpham fork + FN arms), so report the\n        # mean without the high-variance games next to the headline. Printed BEFORE the TOTAL line so callers\n        # that read `tail -n 1` keep getting TOTAL.\n        stable = [r for r in scored if not any(r["game"].startswith(g) for g in HIGH_VARIANCE_GAMES)]\n        if stable and len(stable) < len(scored):\n            ms = lambda k: 100.0 * sum(r[k] for r in stable) / len(stable)\n            print("EX-HIGHVAR (excl %s) over %d games: OFFICIAL(score_cum)=%.3f%%  perfect-retry(score_last)=%.3f%%" % (\n                ",".join(HIGH_VARIANCE_GAMES), len(stable), ms("game_cum"), ms("game_last")))\n        # FI-939 2026-09-28: generalizes EX-HIGHVAR for wide (25-game) runs, where the dominant game need not be\n        # ft09: name the single game whose removal moves the mean most, and the mean without it.\n        if len(scored) >= 3:\n            g_top, share, m_wo = max_influence(scored)\n            print("MAX-INFLUENCE %s: %.1f%% of the total; OFFICIAL(score_cum) without it over %d games=%.3f%%" % (\n                g_top, share, len(scored) - 1, m_wo))\n        print("TOTAL over %d games: OFFICIAL(score_cum)=%.3f%%  perfect-retry(score_last)=%.3f%%  if-every-completed-level-were-1.0=%.3f%%" % (\n            len(scored), m("game_cum"), m("game_last"), m("game_compl")))\n    if out_json:\n        with open(out_json, "w") as f:\n            json.dump(res, f, indent=1)\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main(sys.argv[1:]))\n'
open('duck_score_estimate.py', 'w', encoding='utf-8').write(_SRC)
print('wrote duck_score_estimate.py', len(_SRC), 'chars')


In [ ]:
# embedded verbatim from fi939-agent-rebuild/loadsim_power.py
_SRC = '#!/usr/bin/env python3\n"""FI-939: run-to-run noise and statistical power for loadsim arms (25 games x N passes), from per-pass traces.\n\nWhy (2026-10-08, DUCK 38bg): unanchor (+15 levels, +4%) and D\' (+9) could not be told apart from noise, and every\narm so far is one loadsim run of 4 passes compared with one base run. Before spending the next week\'s GPU quota\nwe need to know, from data we already have, how big an effect a 4-pass loadsim CAN detect, and how many passes an\narm would need to resolve an effect of the size we are seeing.\n\nMethod:\n  - Value per (arm, game, pass) from <game>-<hash>_p<N>_events.jsonl (deduped by basename, shallowest path):\n      levels  = levels completed in that pass                    (default)\n      l2plus  = levels >= 2 completed (what level-transition arms target)\n      score   = official per-game estimate in percent (duck_score_estimate method)\n  - Pass-to-pass noise per game: pooled WITHIN-arm variance s2_g (deviations from each arm\'s own game mean, pooled\n    over arms). Pooling assumes the arms do not change a game\'s noise; it is the only way to get more than 3 degrees\n    of freedom per game from 4-pass runs.\n  - Comparing two arms\' totals (sum over games of the per-game pass mean), with na and nb passes:\n      SD_diff = sqrt(sum_g s2_g * (1/na + 1/nb))\n    The normal approximation is used for the z, the two-sided p, and the 80%-power minimum detectable effect\n    MDE = (1.96 + 0.8416) * SD_diff. Passes needed per arm (equal n) to detect an effect d: n = 2 * 2.8016^2 * S / d^2,\n    S = sum_g s2_g.\nCaveats: levels are discrete and some games have zero variance; a game\'s noise may differ between arms; pass-level\nvalues within one run share a kernel and a server and may not be independent. Read the numbers as an estimate of\norder of magnitude, not a test of record.\n\nUsage: loadsim_power.py label=DIR [label=DIR ...] [--metric levels|l2plus|score] [--passes 4,8,12,16]\n                        [--env-dir DIR] [--json out.json]\n  The first arm is the base; every other arm is compared with it.\nExit: 0 ok | 2 usage | 3 fewer than 2 passes in some arm | 4 an arm has no traces\n"""\nimport glob, json, math, os, re, sys\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, HERE)\nimport duck_score_estimate as S  # noqa: E402\n\nZ_ALPHA, Z_POWER = 1.959964, 0.841621\nK = Z_ALPHA + Z_POWER\nNAME_RE = re.compile(r"^(?P<g>[a-z0-9]+)-[0-9a-f]+_p(?P<p>\\d+)_events\\.jsonl$")\n\n\ndef read_events(path):\n    out = []\n    with open(path, encoding="utf-8", errors="replace") as f:\n        for line in f:\n            try:\n                out.append(json.loads(line))\n            except Exception:\n                pass\n    return out\n\n\ndef value(events, metric, base=None):\n    lv = S.level_actions(events)\n    if metric == "levels":\n        return float(len(lv))\n    if metric == "l2plus":\n        return float(sum(1 for L, _, _ in lv if L >= 2))\n    if metric == "score":\n        if not base:\n            return None\n        cum = {L: S.level_score(base[L - 1], c) for L, c, _ in lv if L <= len(base)}\n        return 100.0 * S.game_score(base, cum)\n    raise ValueError(metric)\n\n\ndef load_arm(path, metric, baselines=None):\n    """{game: {pass: value}} for one arm directory."""\n    paths = [p for p in glob.glob(os.path.join(path, "**", "*_events.jsonl"), recursive=True)\n             if NAME_RE.match(os.path.basename(p))]\n    out = {}\n    for name, p in S.dedupe_by_game(paths).items():\n        m = NAME_RE.match(name)\n        g, ps = m.group("g"), int(m.group("p"))\n        v = value(read_events(p), metric, (baselines or {}).get(g))\n        if v is not None:\n            out.setdefault(g, {})[ps] = v\n    return out\n\n\ndef mean(xs):\n    return sum(xs) / len(xs) if xs else 0.0\n\n\ndef pooled_var(arms, games):\n    """{game: pooled within-arm variance} over the given games; arms: [{game: {pass: v}}]."""\n    out = {}\n    for g in games:\n        ss, df = 0.0, 0\n        for a in arms:\n            xs = list(a[g].values())\n            if len(xs) < 2:\n                continue\n            m = mean(xs)\n            ss += sum((x - m) ** 2 for x in xs)\n            df += len(xs) - 1\n        out[g] = ss / df if df else 0.0\n    return out\n\n\ndef sd_diff(s2, na, nb):\n    return math.sqrt(sum(s2.values()) * (1.0 / na + 1.0 / nb))\n\n\ndef p_two_sided(z):\n    return math.erfc(abs(z) / math.sqrt(2.0))\n\n\ndef passes_needed(s2, d):\n    """Equal passes per arm to detect an effect d (in total units) at alpha .05 two-sided, power .8."""\n    if not d:\n        return None\n    return math.ceil(2.0 * K * K * sum(s2.values()) / (d * d))\n\n\ndef analyze(specs, metric="levels", passes=(4, 8, 12, 16), baselines=None):\n    arms = [(lab, load_arm(d, metric, baselines)) for lab, d in specs]\n    for lab, a in arms:\n        if not a:\n            return {"error": "no traces", "arm": lab}\n    games = sorted(set.intersection(*[set(a) for _, a in arms]))\n    nmin = {lab: min(len(a[g]) for g in games) for lab, a in arms} if games else {}\n    for lab, n in nmin.items():\n        if n < 2:\n            return {"error": "fewer than 2 passes", "arm": lab}\n    s2 = pooled_var([a for _, a in arms], games)\n    totals = {lab: sum(mean(list(a[g].values())) for g in games) for lab, a in arms}\n    base_lab = arms[0][0]\n    sum_tot = totals[base_lab] or 1.0\n    noisiest = sorted(s2.items(), key=lambda kv: -kv[1])[:5]\n    S2 = sum(s2.values())\n    mde = [{"passes": n, "sd_diff": math.sqrt(S2 * 2.0 / n), "mde80": K * math.sqrt(S2 * 2.0 / n),\n            "mde80_pct": 100.0 * K * math.sqrt(S2 * 2.0 / n) / sum_tot} for n in passes]\n    comps = []\n    for lab, a in arms[1:]:\n        na, nb = nmin[base_lab], nmin[lab]\n        sd = sd_diff(s2, na, nb)\n        d = totals[lab] - totals[base_lab]\n        z = d / sd if sd else 0.0\n        comps.append({"arm": lab, "diff": d, "diff_pct": 100.0 * d / sum_tot, "sd_diff": sd, "z": z,\n                      "p": p_two_sided(z) if sd else 1.0, "passes_needed": passes_needed(s2, d)})\n    return {"metric": metric, "games": len(games), "passes": nmin, "totals": totals, "sum_var": S2,\n            "noisiest": noisiest, "mde": mde, "compare": comps, "base": base_lab}\n\n\ndef render(r):\n    lines = ["metric=%s games=%d base=%s passes=%s" % (r["metric"], r["games"], r["base"],\n             ",".join("%s:%d" % kv for kv in r["passes"].items()))]\n    lines.append("TOTALS (sum over games of per-game pass mean): " +\n                 "  ".join("%s=%.2f" % kv for kv in r["totals"].items()))\n    lines.append("NOISE: sum of pooled per-game pass variance = %.3f; noisiest games: %s" % (\n        r["sum_var"], ", ".join("%s %.2f" % kv for kv in r["noisiest"])))\n    for m in r["mde"]:\n        lines.append("MDE80 at %2d passes per arm: SD_diff=%.2f  MDE=%.2f (%.1f%% of base total)" % (\n            m["passes"], m["sd_diff"], m["mde80"], m["mde80_pct"]))\n    for c in r["compare"]:\n        lines.append("COMPARE %s - %s: diff=%+.2f (%+.1f%%)  SD=%.2f  z=%+.2f  p=%.3f  passes/arm for 80%% power: %s" % (\n            c["arm"], r["base"], c["diff"], c["diff_pct"], c["sd_diff"], c["z"], c["p"],\n            c["passes_needed"] if c["passes_needed"] is not None else "n/a"))\n    return "\\n".join(lines)\n\n\ndef main(argv):\n    metric, passes, env_dir, out_json, specs = "levels", (4, 8, 12, 16), S.ENV_DIR, None, []\n    i = 0\n    while i < len(argv):\n        a = argv[i]\n        if a == "--metric" and i + 1 < len(argv):\n            metric = argv[i + 1]; i += 2\n        elif a == "--passes" and i + 1 < len(argv):\n            try:\n                passes = tuple(int(x) for x in argv[i + 1].split(",") if x)\n            except ValueError:\n                print(__doc__); return 2\n            i += 2\n        elif a == "--env-dir" and i + 1 < len(argv):\n            env_dir = argv[i + 1]; i += 2\n        elif a == "--json" and i + 1 < len(argv):\n            out_json = argv[i + 1]; i += 2\n        elif "=" in a and not a.startswith("--"):\n            lab, d = a.split("=", 1); specs.append((lab, d)); i += 1\n        else:\n            print(__doc__); return 2\n    if len(specs) < 2 or metric not in ("levels", "l2plus", "score") or not passes or min(passes) < 1:\n        print(__doc__); return 2\n    bl = S.load_baselines(env_dir) if metric == "score" else None\n    r = analyze(specs, metric, passes, bl)\n    if r.get("error") == "no traces":\n        print("NO_TRACES: %s" % r["arm"]); return 4\n    if r.get("error"):\n        print("TOO_FEW_PASSES: %s" % r["arm"]); return 3\n    print(render(r))\n    if out_json:\n        with open(out_json, "w") as f:\n            json.dump(r, f, indent=1)\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main(sys.argv[1:]))\n'
open('loadsim_power.py', 'w', encoding='utf-8').write(_SRC)
print('wrote loadsim_power.py', len(_SRC), 'chars')


In [ ]:
# embedded verbatim from fi939-agent-rebuild/control_mix.py
_SRC = '#!/usr/bin/env python3\n"""FI-939: control-family coverage per level from environment event logs (paper table T4, DUCK 38bc).\n\nWhy: Figure 1 shows one lf52 pass spending ~45 clicks on level 2 before trying the movement keys. This measures the\npattern across all games and passes, from ground truth only (events.jsonl). For each (game, pass, level) segment:\n  - families used on the level (move ACTION1-4 / act ACTION5 / click ACTION6 / undo ACTION7);\n  - families the GAME has (used anywhere in that game, any pass, any level) but this level never tried;\n  - actions before the level\'s second distinct family appeared (None if only one family was used);\n  - last_family_at: position (fraction of the level\'s actions) at which the LAST of the game\'s families was\n    first tried on this level (1.0 if one was never tried).\nThen it compares won and failed segments, split L1 vs L2+, and (WITHIN) per game: only games with >= 2 families,\nL2+ segments with >= 10 actions, comparing mean last_family_at of won vs failed segments of the same game.\nThat within-game view matters: a pooled comparison mixes games that need one family with games that need several.\nRESET is ignored (it is not a control).\nUsage: control_mix.py TRACE_DIR [TRACE_DIR ...] [--json OUT]\n"""\nimport glob, json, os, re, sys\nfrom collections import defaultdict\n\nFAM = {"ACTION1": "move", "ACTION2": "move", "ACTION3": "move", "ACTION4": "move",\n       "ACTION5": "act", "ACTION6": "click", "ACTION7": "undo"}\n\n\ndef segments(acts):\n    """Split one pass\'s action events into level segments: [{level, won, fams:[...]}] in order."""\n    segs, cur, lvl = [], [], 1\n    for e in acts:\n        f = FAM.get(str(e.get("action_name") or "").upper())\n        if f:\n            cur.append(f)\n        # FI-939 2026-10-08 (38bg): the game-winning action has level_completed False and state WIN, so the\n        # final level of a won game was counted as a FAILED segment (base run: 369 vs 402 events-score levels).\n        if e.get("level_completed") or str(e.get("state") or "").upper() == "WIN":\n            segs.append({"level": lvl, "won": True, "fams": cur})\n            cur, lvl = [], lvl + 1\n    if cur:\n        segs.append({"level": lvl, "won": False, "fams": cur})\n    return segs\n\n\ndef second_family_at(fams):\n    seen = []\n    for i, f in enumerate(fams):\n        if f not in seen:\n            seen.append(f)\n            if len(seen) == 2:\n                return i\n    return None\n\n\ndef last_family_at(fams, game_families):\n    n = len(fams)\n    if not n:\n        return 1.0\n    return max((fams.index(f) / n if f in fams else 1.0) for f in game_families) if game_families else 0.0\n\n\ndef within_game(rows, min_actions=10):\n    """{game: {won: [..], failed: [..]}} of last_family_at over L2+ segments, games with >= 2 families."""\n    out = defaultdict(lambda: {"won": [], "failed": []})\n    for r in rows:\n        if r["level"] < 2 or r["n"] < min_actions or r["game_families"] < 2:\n            continue\n        out[r["game"].split(":")[-1].split("-")[0]]["won" if r["won"] else "failed"].append(r["last_family_at"])\n    return dict(out)\n\n\ndef load(dirs):\n    """{(game, pass): [action events]} from *_pN_events.jsonl, one file per basename (shallowest)."""\n    out = {}\n    for d in dirs:\n        paths = sorted(glob.glob(os.path.join(d, "**", "*_p*_events.jsonl"), recursive=True),\n                       key=lambda x: (x.count(os.sep), x))\n        for p in paths:\n            m = re.match(r"(.+?)_(p\\d+)_events\\.jsonl$", os.path.basename(p))\n            if not m:\n                continue\n            key = (os.path.basename(d.rstrip("/")) + ":" + m.group(1), m.group(2))\n            if key in out:\n                continue\n            acts = []\n            with open(p) as f:\n                for line in f:\n                    try:\n                        e = json.loads(line)\n                    except Exception:\n                        continue\n                    if e.get("type") == "action":\n                        acts.append(e)\n            out[key] = acts\n    return out\n\n\ndef analyse(runs):\n    game_fams = defaultdict(set)\n    segs = []\n    for (game, pas), acts in runs.items():\n        for s in segments(acts):\n            s.update(game=game, pas=pas)\n            segs.append(s)\n            game_fams[game].update(s["fams"])\n    rows = []\n    for s in segs:\n        used = set(s["fams"])\n        rows.append({"game": s["game"], "pass": s["pas"], "level": s["level"], "won": s["won"], "n": len(s["fams"]),\n                     "used": sorted(used), "untried": sorted(game_fams[s["game"]] - used),\n                     "second_at": second_family_at(s["fams"]),\n                     "game_families": len(game_fams[s["game"]]),\n                     "last_family_at": round(last_family_at(s["fams"], game_fams[s["game"]]), 4)})\n    return rows\n\n\ndef summarise(rows, min_actions=10):\n    """Groups (L1|L2+, won|failed) over segments with >= min_actions actions (short segments say little)."""\n    out = {}\n    for lv in ("L1", "L2+"):\n        for won in (True, False):\n            g = [r for r in rows if (r["level"] == 1) == (lv == "L1") and r["won"] == won and r["n"] >= min_actions]\n            if not g:\n                continue\n            k = "%s %s" % (lv, "won" if won else "failed")\n            multi = [r for r in g if len(set(r["used"])) >= 1]\n            sec = [r["second_at"] for r in g if r["second_at"] is not None]\n            out[k] = {"segments": len(g),\n                      "share_with_untried_family": sum(1 for r in g if r["untried"]) / len(g),\n                      "mean_families_used": sum(len(r["used"]) for r in multi) / len(multi),\n                      "share_single_family": sum(1 for r in g if len(r["used"]) == 1) / len(g),\n                      "median_actions_before_2nd_family": sorted(sec)[len(sec) // 2] if sec else None}\n    return out\n\n\ndef main(argv):\n    out_json, dirs = None, []\n    i = 0\n    while i < len(argv):\n        if argv[i] == "--json":\n            out_json = argv[i + 1]; i += 2\n        else:\n            dirs.append(argv[i]); i += 1\n    if not dirs:\n        print(__doc__); return 2\n    rows = analyse(load(dirs))\n    summ = summarise(rows)\n    print("%-12s %8s %22s %18s %16s %22s" % ("group", "segments", "has_untried_family", "families_used", "single_family", "median_acts_to_2nd_fam"))\n    for k, v in summ.items():\n        print("%-12s %8d %21.0f%% %18.2f %15.0f%% %22s" % (k, v["segments"], 100 * v["share_with_untried_family"],\n              v["mean_families_used"], 100 * v["share_single_family"], v["median_actions_before_2nd_family"]))\n    wg = within_game(rows)\n    mean = lambda xs: sum(xs) / len(xs)\n    print("WITHIN-GAME last_family_at (L2+, >=10 actions, games with >=2 families): game nW nF won failed")\n    both = []\n    for g in sorted(wg):\n        w, f = wg[g]["won"], wg[g]["failed"]\n        print("  %-6s %3d %3d %6s %6s" % (g, len(w), len(f), "%.2f" % mean(w) if w else "-", "%.2f" % mean(f) if f else "-"))\n        if w and f:\n            both.append((mean(w), mean(f)))\n    if both:\n        print("WITHIN-GAME over %d games with both: mean won=%.2f failed=%.2f; failed later in %d/%d games" % (\n            len(both), mean([x for x, _ in both]), mean([y for _, y in both]), sum(y > x for x, y in both), len(both)))\n    if out_json:\n        json.dump({"rows": rows, "summary": summ, "within_game": wg}, open(out_json, "w"), indent=1)\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main(sys.argv[1:]))\n'
open('control_mix.py', 'w', encoding='utf-8').write(_SRC)
print('wrote control_mix.py', len(_SRC), 'chars')


In [ ]:
# embedded verbatim from fi939-agent-rebuild/probe_audit.py
_SRC = '#!/usr/bin/env python3\n"""FI-939: audit what probe_controls() actually did in a ctrlprobe run (DUCK 38bh).\n\nWhy: the ctrlprobe arm (38av) gives the model a controls diagnostic that tries keys, clicks one object per kind,\nand re-tries keys after an effective click. An arm readout counts levels; it cannot say whether the probe ran as\ndesigned. Reading the first probe table in the 2026-10-08 run showed `probe_controls: 1 actions` followed by every\nlater row `REFUSED StaleStateActionError`: the M2 sandbox refuses any action in a snippet after a no-op, so one\ninert key ended the whole probe. This tool measures how often that happened, over every game and pass.\n\nMethod: each `analysis` event\'s transcript carries the turn history, so the same tool result repeats across\nevents. Per (game, pass) we keep each distinct probe table once (header line plus its rows). A table is\n  BLOCKED      if it executed <= 1 action and has >= 2 rows refused by the stale-state guard;\n  PARTIAL      if any row was refused but it is not BLOCKED;\n  CLEAN        otherwise.\nWe also count distinct python tool-call code blocks containing `probe_controls(` (calls whose table never printed,\ne.g. because a BaseException guard ended the snippet, show up as calls without a table).\nCaveat: two byte-identical tables in one pass are counted once.\n\nUsage: probe_audit.py TRACE_DIR [--games g1,g2] [--json out.json]\nExit: 0 ok | 2 usage | 4 no events files\n"""\nimport glob, json, os, re, sys\nfrom collections import Counter, defaultdict\n\nNAME_RE = re.compile(r"^(?P<g>[a-z0-9]+)-[0-9a-f]+_p(?P<p>\\d+)_events\\.jsonl$")\nHEAD_RE = re.compile(r"^probe_controls: (\\d+) actions(?: \\(stopped: ([^)]*)\\))?\\s*$")\nROW_RE = re.compile(r"^  (.+?)\\s+-> (.*)$")\nCALL_RE = re.compile(r"\\[TOOL CALL: python\\]\\s*<tool_call>(.*?)</tool_call>", re.S)\n\n\ndef parse_tables(text):\n    """Distinct probe tables in one transcript: [(header_line, [(label, effect), ...])]."""\n    out = []\n    lines = text.split("\\n")\n    i = 0\n    while i < len(lines):\n        m = HEAD_RE.match(lines[i])\n        if not m:\n            i += 1\n            continue\n        rows = []\n        j = i + 1\n        while j < len(lines):\n            r = ROW_RE.match(lines[j])\n            if not r:\n                break\n            rows.append((r.group(1).strip(), r.group(2).strip()))\n            j += 1\n        out.append((lines[i].strip(), rows))\n        i = j\n    return out\n\n\ndef classify(header, rows):\n    m = HEAD_RE.match(header)\n    used = int(m.group(1)) if m else 0\n    refused = Counter()\n    for _, eff in rows:\n        if eff.startswith("REFUSED"):\n            refused[eff.split(" ", 1)[1] if " " in eff else "?"] += 1\n    stale = refused.get("StaleStateActionError", 0)\n    if used <= 1 and stale >= 2:\n        kind = "BLOCKED"\n    elif refused:\n        kind = "PARTIAL"\n    else:\n        kind = "CLEAN"\n    changed = sum(1 for _, eff in rows if not eff.startswith("REFUSED") and not eff.startswith("nothing"))\n    return {"used": used, "kind": kind, "refused": dict(refused), "rows": len(rows), "changed": changed}\n\n\ndef audit_file(path):\n    tables, calls = {}, set()\n    with open(path, encoding="utf-8", errors="replace") as f:\n        for line in f:\n            try:\n                e = json.loads(line)\n            except Exception:\n                continue\n            if e.get("type") != "analysis":\n                continue\n            t = str(e.get("transcript") or "")\n            if "probe_controls" not in t:\n                continue\n            for code in CALL_RE.findall(t):\n                if "probe_controls(" in code and "def probe_controls" not in code:\n                    calls.add(code.strip())\n            for h, rows in parse_tables(t):\n                tables.setdefault((h, tuple(rows)), classify(h, rows))\n    return {"calls": len(calls), "tables": list(tables.values())}\n\n\ndef audit(trace_dir, games=None):\n    paths = [p for p in glob.glob(os.path.join(trace_dir, "**", "*_events.jsonl"), recursive=True)\n             if NAME_RE.match(os.path.basename(p))]\n    best = {}\n    for p in sorted(paths, key=lambda x: (x.count(os.sep), x)):\n        best.setdefault(os.path.basename(p), p)\n    per = defaultdict(lambda: {"calls": 0, "tables": 0, "BLOCKED": 0, "PARTIAL": 0, "CLEAN": 0,\n                               "actions": 0, "changed_rows": 0, "refused": Counter()})\n    for name, p in sorted(best.items()):\n        g = NAME_RE.match(name).group("g")\n        if games and g not in games:\n            continue\n        r = audit_file(p)\n        d = per[g]\n        d["calls"] += r["calls"]\n        for t in r["tables"]:\n            d["tables"] += 1\n            d[t["kind"]] += 1\n            d["actions"] += t["used"]\n            d["changed_rows"] += t["changed"]\n            d["refused"].update(t["refused"])\n    return per, len(best)\n\n\ndef render(per):\n    lines = ["%-6s %5s %6s %7s %7s %5s %7s %7s  refused" % ("game", "calls", "tables", "BLOCKED", "PARTIAL", "CLEAN",\n                                                             "actions", "changed")]\n    tot = defaultdict(int)\n    ref = Counter()\n    for g in sorted(per):\n        d = per[g]\n        lines.append("%-6s %5d %6d %7d %7d %5d %7d %7d  %s" % (g, d["calls"], d["tables"], d["BLOCKED"], d["PARTIAL"],\n                     d["CLEAN"], d["actions"], d["changed_rows"], dict(d["refused"]) or "-"))\n        for k in ("calls", "tables", "BLOCKED", "PARTIAL", "CLEAN", "actions", "changed_rows"):\n            tot[k] += d[k]\n        ref.update(d["refused"])\n    lines.append("%-6s %5d %6d %7d %7d %5d %7d %7d  %s" % ("TOTAL", tot["calls"], tot["tables"], tot["BLOCKED"],\n                 tot["PARTIAL"], tot["CLEAN"], tot["actions"], tot["changed_rows"], dict(ref) or "-"))\n    if tot["tables"]:\n        lines.append("BLOCKED share %.0f%% of tables; mean executed actions per table %.1f" % (\n            100.0 * tot["BLOCKED"] / tot["tables"], tot["actions"] / tot["tables"]))\n    return "\\n".join(lines), dict(tot), ref\n\n\ndef main(argv):\n    games, out_json, args = None, None, []\n    i = 0\n    while i < len(argv):\n        if argv[i] == "--games" and i + 1 < len(argv):\n            games = set(x for x in argv[i + 1].split(",") if x); i += 2\n        elif argv[i] == "--json" and i + 1 < len(argv):\n            out_json = argv[i + 1]; i += 2\n        elif argv[i].startswith("--"):\n            print(__doc__); return 2\n        else:\n            args.append(argv[i]); i += 1\n    if len(args) != 1:\n        print(__doc__); return 2\n    per, n = audit(args[0], games)\n    if not n:\n        print("NO_EVENTS: %s" % args[0]); return 4\n    text, tot, ref = render(per)\n    print(text)\n    if out_json:\n        with open(out_json, "w") as f:\n            json.dump({"per_game": {g: dict(d, refused=dict(d["refused"])) for g, d in per.items()},\n                       "total": tot, "refused": dict(ref)}, f, indent=1)\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main(sys.argv[1:]))\n'
open('probe_audit.py', 'w', encoding='utf-8').write(_SRC)
print('wrote probe_audit.py', len(_SRC), 'chars')


In [ ]:
# embedded verbatim from fi939-agent-rebuild/targeted_gate.py
_SRC = '#!/usr/bin/env python3\n"""FI-939: evaluate the pre-registered 38bi gate for a targeted multi-pass arm against its same-shape base.\n\nWhy (DUCK 38bi): targeted arms (ctrlprobe2, inventory) run 6 games x 16 passes next to a 6 x 16 base. The reading\nwas written down before any data; this applies it mechanically, so the verdict cannot drift toward whatever the\nnumbers turn out to be.\n\n  1. MECHANISM, per target game: count passes that cleared level 2 (levels >= 2) in base and arm. A target game\n     PASSES when the arm clears L2 in at least `--min-gain` (default 4) more passes than the base AND the base\n     cleared it in at most `--base-max` (default 1) passes. The arm passes MECHANISM if any target game passes.\n  2. COST, over the cost games together: total levels (sum of per-game pass means) must not fall by more than the\n     80%-power MDE of that difference (loadsim_power pooled variance over the two runs, their own pass counts).\n     A drop larger than the MDE is FAIL; any smaller change is PASS (not "no effect", just not a detected cost).\n  3. VERDICT: GATE_PASS (1 and 2) -> earns a 25 x 4 confirmation run before any hidden slot; otherwise GATE_FAIL.\n\nUsage: targeted_gate.py BASE_DIR ARM_DIR [--targets lf52,bp35,sk48] [--cost r11l,sc25,ar25]\n                        [--min-gain 4] [--base-max 1] [--json out.json]\nExit: 0 evaluated (verdict in output) | 2 usage | 4 a target or cost game missing from either run\n"""\nimport argparse, json, math, os, sys\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, HERE)\nimport loadsim_power as P  # noqa: E402\n\n\ndef l2_passes(arm, game):\n    return sum(1 for v in arm[game].values() if v >= 2)\n\n\ndef evaluate(base, arm, targets, cost, min_gain=4, base_max=1):\n    missing = [g for g in list(targets) + list(cost) if g not in base or g not in arm]\n    if missing:\n        return {"error": "missing", "games": missing}\n    mech = []\n    for g in targets:\n        b, a = l2_passes(base, g), l2_passes(arm, g)\n        mech.append({"game": g, "base_l2": b, "arm_l2": a, "base_n": len(base[g]), "arm_n": len(arm[g]),\n                     "pass": (a - b) >= min_gain and b <= base_max})\n    s2 = P.pooled_var([base, arm], list(cost))\n    nb = min(len(base[g]) for g in cost)\n    na = min(len(arm[g]) for g in cost)\n    tb = sum(P.mean(list(base[g].values())) for g in cost)\n    ta = sum(P.mean(list(arm[g].values())) for g in cost)\n    sd = math.sqrt(sum(s2.values()) * (1.0 / nb + 1.0 / na)) if nb and na else 0.0\n    mde = P.K * sd\n    diff = ta - tb\n    cost_pass = not (diff < 0 and -diff > mde)\n    mech_pass = any(m["pass"] for m in mech)\n    return {"mechanism": mech, "mechanism_pass": mech_pass,\n            "cost": {"games": list(cost), "base_total": tb, "arm_total": ta, "diff": diff, "mde80": mde,\n                     "pass": cost_pass},\n            "verdict": "GATE_PASS" if (mech_pass and cost_pass) else "GATE_FAIL"}\n\n\ndef render(r):\n    lines = ["MECHANISM (L2 cleared in how many passes; pass = arm - base >= gain and base <= base-max):"]\n    for m in r["mechanism"]:\n        lines.append("  %-5s base %2d/%-2d  arm %2d/%-2d  %s" % (m["game"], m["base_l2"], m["base_n"], m["arm_l2"],\n                                                            m["arm_n"], "PASS" if m["pass"] else "-"))\n    c = r["cost"]\n    lines.append("COST (%s total levels per pass-set): base %.2f arm %.2f diff %+.2f, MDE80 %.2f -> %s" % (\n        ",".join(c["games"]), c["base_total"], c["arm_total"], c["diff"], c["mde80"], "PASS" if c["pass"] else "FAIL"))\n    lines.append("%s (mechanism %s, cost %s)" % (r["verdict"], "PASS" if r["mechanism_pass"] else "FAIL",\n                                                 "PASS" if c["pass"] else "FAIL"))\n    return "\\n".join(lines)\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    ap.add_argument("base")\n    ap.add_argument("arm")\n    ap.add_argument("--targets", default="lf52,bp35,sk48")\n    ap.add_argument("--cost", default="r11l,sc25,ar25")\n    ap.add_argument("--min-gain", type=int, default=4)\n    ap.add_argument("--base-max", type=int, default=1)\n    ap.add_argument("--json")\n    a = ap.parse_args(argv)\n    targets = [g for g in a.targets.split(",") if g]\n    cost = [g for g in a.cost.split(",") if g]\n    if not targets or not cost or a.min_gain < 1 or a.base_max < 0:\n        print(__doc__); return 2\n    r = evaluate(P.load_arm(a.base, "levels"), P.load_arm(a.arm, "levels"), targets, cost, a.min_gain, a.base_max)\n    if r.get("error"):\n        print("GATE_MISSING: %s" % ",".join(r["games"])); return 4\n    print(render(r))\n    if a.json:\n        with open(a.json, "w") as f:\n            json.dump(r, f, indent=1)\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'
open('targeted_gate.py', 'w', encoding='utf-8').write(_SRC)
print('wrote targeted_gate.py', len(_SRC), 'chars')


In [ ]:
# embedded verbatim from fi939-agent-rebuild/make_fig2.py
_SRC = '#!/usr/bin/env python3\n"""FI-939 paper Figure 2: how large an effect one loadsim can detect (DUCK 38bg/38bi).\n\nPanel A: the 80%-power minimum detectable effect (levels won, % of the base total) against passes per arm, for the\n25-game design, computed from pooled pass-to-pass variance (loadsim_power.py). Measured arm effects are drawn as\nreference lines so the reader sees at once which ones a single 4-pass run could have confirmed.\nPanel B: where that noise lives - per-game pooled pass variance, largest first.\n\nUsage: make_fig2.py base=DIR arm=DIR [...] [--out paper/fig2_power] [--max-passes 32]\n  The first spec is the base; every other spec is drawn as a measured effect (diff vs base).\nExit: 0 ok | 2 usage | 4 no traces\n"""\nimport argparse, math, os, sys\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, HERE)\nimport loadsim_power as P  # noqa: E402\n\nINK, MUTED, GRID, HUE = "#1f1f1e", "#6b6a63", "#d9d8d2", "#2a78d6"\n\n\ndef compute(specs, max_passes=32):\n    """Pure: the figure\'s numbers. Returns dict with curve [(n, mde_pct)], effects [(label, pct)], variances."""\n    r = P.analyze(specs, "levels", tuple(range(2, max_passes + 1, 2)))\n    if r.get("error"):\n        return r\n    base_tot = r["totals"][r["base"]] or 1.0\n    curve = [(m["passes"], m["mde80_pct"]) for m in r["mde"]]\n    effects = [(c["arm"], c["diff_pct"]) for c in r["compare"]]\n    arms = [P.load_arm(d, "levels") for _, d in specs]\n    games = sorted(set.intersection(*[set(a) for a in arms]))\n    var = sorted(P.pooled_var(arms, games).items(), key=lambda kv: -kv[1])\n    return {"curve": curve, "effects": effects, "variances": var, "base_total": base_tot, "games": len(games),\n            "passes": r["passes"]}\n\n\ndef passes_for(curve, pct):\n    """Smallest passes on the curve whose MDE is <= pct (None if none)."""\n    for n, m in curve:\n        if m <= pct:\n            return n\n    return None\n\n\ndef draw(d, out):\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1.15, 1]})\n    for ax in (a1, a2):\n        for s in ("top", "right"):\n            ax.spines[s].set_visible(False)\n        for s in ("left", "bottom"):\n            ax.spines[s].set_color(MUTED)\n        ax.tick_params(colors=MUTED, labelsize=9)\n        ax.grid(axis="y", color=GRID, linewidth=0.8)\n        ax.set_axisbelow(True)\n    xs = [n for n, _ in d["curve"]]\n    ys = [m for _, m in d["curve"]]\n    a1.plot(xs, ys, color=HUE, linewidth=2)\n    a1.scatter([4], [dict(d["curve"]).get(4, ys[0])], s=40, color=HUE, zorder=3, edgecolor="white", linewidth=2)\n    a1.annotate("one 4-pass run: %.0f%%" % dict(d["curve"]).get(4, ys[0]), (4, dict(d["curve"]).get(4, ys[0])),\n                xytext=(10, 4), textcoords="offset points", color=INK, fontsize=9)\n    seen = []\n    for lab, pct in d["effects"]:\n        y = abs(pct)\n        a1.axhline(y, color=MUTED, linewidth=1, linestyle=(0, (4, 3)))\n        dy = 3 if any(abs(y - s) < 0.6 for s in seen) else -11\n        seen.append(y)\n        n = passes_for(d["curve"], y)\n        a1.annotate("%s %+.1f%%: %s" % (lab, pct, ("about %d passes per arm" % n) if n else\n                                           "needs > %d passes per arm" % xs[-1]),\n                    (xs[-1], y), xytext=(-4, dy), textcoords="offset points", ha="right", color=MUTED, fontsize=8.5,\n                    bbox=dict(boxstyle="square,pad=0.1", facecolor="white", edgecolor="none"))\n    a1.set_xlim(0, xs[-1] + 1)\n    a1.set_ylim(0, max(ys) * 1.08)\n    a1.set_xlabel("passes per arm (25 games)", color=INK, fontsize=10)\n    a1.set_ylabel("minimum detectable effect, % of levels won", color=INK, fontsize=10)\n    a1.set_title("A. What one loadsim can confirm (80% power)", color=INK, fontsize=11, loc="left")\n    top = d["variances"][:12]\n    a2.bar(range(len(top)), [v for _, v in top], color=HUE, width=0.7)\n    a2.set_xticks(range(len(top)))\n    a2.set_xticklabels([g for g, _ in top], rotation=45, ha="right", color=INK, fontsize=9)\n    a2.set_ylabel("pass-to-pass variance of levels won", color=INK, fontsize=10)\n    a2.set_title("B. Where the noise lives (12 of %d games)" % d["games"], color=INK, fontsize=11, loc="left")\n    fig.text(0.01, 0.01, "Pooled within-arm variance over %s; normal approximation; passes in one run may not be "\n             "independent." % ", ".join("%s (%d)" % kv for kv in d["passes"].items()), color=MUTED, fontsize=7.5)\n    fig.tight_layout(rect=(0, 0.04, 1, 1))\n    os.makedirs(os.path.dirname(out) or ".", exist_ok=True)\n    fig.savefig(out + ".png", dpi=200, facecolor="white")\n    fig.savefig(out + ".svg", facecolor="white")\n    plt.close(fig)\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    ap.add_argument("specs", nargs="+")\n    ap.add_argument("--out", default=os.path.join(HERE, "paper", "fig2_power"))\n    ap.add_argument("--max-passes", type=int, default=32)\n    a = ap.parse_args(argv)\n    specs = []\n    for s in a.specs:\n        if "=" not in s:\n            print(__doc__); return 2\n        specs.append(tuple(s.split("=", 1)))\n    if len(specs) < 2 or a.max_passes < 4:\n        print(__doc__); return 2\n    d = compute(specs, a.max_passes)\n    if d.get("error"):\n        print("NO_TRACES_OR_PASSES: %s" % d.get("arm")); return 4\n    draw(d, a.out)\n    print("FIG2_OK %s.png  MDE at 4 passes %.1f%%; effects %s" % (\n        a.out, dict(d["curve"]).get(4, float("nan")), ", ".join("%s %+.1f%%" % e for e in d["effects"])))\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'
open('make_fig2.py', 'w', encoding='utf-8').write(_SRC)
print('wrote make_fig2.py', len(_SRC), 'chars')


In [ ]:
# embedded verbatim from fi939-agent-rebuild/replay_claims.py
_SRC = '#!/usr/bin/env python3\n"""FI-939: test DUCK 38ap H1/H2 on recorded M2 traces - how much evidence stands behind the model\'s "this control\ndoes nothing" claims, and how often such a claim is later contradicted on the same level.\n\nInput: *_requests.jsonl files (one per game pass). Each request carries the (trimmed) conversation history, so the\nunion of all requests, in first-seen order, reconstructs the pass chronologically.\n- level: from user/tool text "Current state: step N, level L".\n- per-action verdicts: tool stdout lines "[action] NAME(...) -> effect" / "-> NO-OP".\n- claims: assistant reasoning sentences that pair a control (UP/DOWN/LEFT/RIGHT, arrows, MOUSE/click, UNDO, SPACE)\n  with a negative word (inert, no-op, does nothing, no effect, useless).\n\nFor the first claim about each control on each level it records: tries of that control on that level before the\nclaim and how many were no-ops, whether that control had already produced an effect (claim contradicts evidence),\nwhether it produces an effect LATER on the same level (claim falsified) and how many actions later, and whether the\nsentence appeals to an earlier level ("as in level 1"), which is H2.\n\nUsage: replay_claims.py FILE_OR_DIR [...] [--json OUT]\n"""\nimport argparse, glob, hashlib, json, os, re, sys\nfrom collections import defaultdict\n\nDIRS = ("UP", "DOWN", "LEFT", "RIGHT")\nSTATE = re.compile(r"Current state: step (\\d+), level (\\d+)")\nVERDICT = re.compile(r"\\[action\\] ([A-Z0-9_]+)(?:\\([^)]*\\))? -> (effect|NO-OP|GAME OVER|LEVEL COMPLETED|RUN COMPLETE)")\nNEG = r"(inert|no-?ops?\\b|does nothing|do nothing|did nothing|no effect|has no effect|useless|non-functional)"\nCTRL = r"(UP|DOWN|LEFT|RIGHT|arrows?|arrow keys|directional(?: keys)?|MOUSE|clicks?|clicking|UNDO|SPACE(?:BAR)?|ACTION[1-7])"\nCLAIM = re.compile(r"[^.\\n]*\\b" + CTRL + r"\\b[^.\\n]{0,60}?\\b" + NEG + r"[^.\\n]*", re.I)\nEARLIER = re.compile(r"(as in|like in|same as in|carried from|from) (the )?(level \\d|previous level|earlier level|last level|level one)", re.I)\n\n\ndef control_of(word):\n    w = word.upper()\n    if w.startswith("ARROW") or w.startswith("DIRECTIONAL"):\n        return "DIRS"\n    if w.startswith("CLICK") or w == "MOUSE" or w == "ACTION6":\n        return "MOUSE"\n    if w.startswith("SPACE") or w == "ACTION5":\n        return "SPACE"\n    if w == "ACTION7":\n        return "UNDO"\n    return {"ACTION1": "UP", "ACTION2": "DOWN", "ACTION3": "LEFT", "ACTION4": "RIGHT"}.get(w, w)\n\n\ndef text_of(content):\n    if isinstance(content, str):\n        return content\n    return "\\n".join(p.get("text", "") for p in content or [] if isinstance(p, dict) and p.get("type") == "text")\n\n\ndef chronological(path):\n    """Unique history messages of a pass in first-seen order: [(role, text, reasoning)]."""\n    seen, out = set(), []\n    with open(path, encoding="utf-8", errors="replace") as f:\n        for line in f:\n            if not line.strip():\n                continue\n            try:\n                r = json.loads(line)\n            except ValueError:\n                continue\n            if r.get("event") not in (None, "request"):\n                continue\n            for m in r.get("messages", [])[1:]:\n                rc = str(m.get("reasoning_content") or m.get("reasoning") or "")\n                tx = text_of(m.get("content"))\n                if m.get("role") == "assistant":\n                    for tc in m.get("tool_calls") or []:\n                        tx += "\\n" + str((tc.get("function") or {}).get("arguments", ""))\n                k = hashlib.sha1((m.get("role", "") + "\\x00" + tx + "\\x00" + rc).encode()).hexdigest()\n                if k in seen:\n                    continue\n                seen.add(k)\n                out.append((m.get("role"), tx, rc))\n    return out\n\n\ndef analyse(msgs):\n    level, n_action = 1, 0\n    tries = defaultdict(lambda: defaultdict(lambda: [0, 0]))      # level -> control -> [tries, noops]\n    effects = defaultdict(lambda: defaultdict(list))               # level -> control -> [action index of effect]\n    claims = []\n    seen_claim = set()\n    for role, tx, rc in msgs:\n        for m in STATE.finditer(tx):\n            level = int(m.group(2))\n        if role == "tool":\n            for m in VERDICT.finditer(tx):\n                n_action += 1\n                c = control_of(m.group(1))\n                t = tries[level][c]\n                t[0] += 1\n                if m.group(2) == "NO-OP":\n                    t[1] += 1\n                elif m.group(2) == "effect":\n                    effects[level][c].append(n_action)\n        elif role == "assistant" and rc:\n            for m in CLAIM.finditer(rc):\n                c = control_of(m.group(1))\n                if (level, c) in seen_claim:\n                    continue\n                seen_claim.add((level, c))\n                ctrls = DIRS if c == "DIRS" else (c,)\n                before = sum(tries[level][x][0] for x in ctrls)\n                noops = sum(tries[level][x][1] for x in ctrls)\n                untried = [x for x in ctrls if tries[level][x][0] == 0]\n                claims.append({"level": level, "control": c, "at_action": n_action, "tries_before": before,\n                               "noops_before": noops, "untried": untried,\n                               "effect_before": any(effects[level][x] for x in ctrls),\n                               "earlier_level": bool(EARLIER.search(m.group(0))),\n                               "sentence": " ".join(m.group(0).split())[:220], "_ctrls": ctrls})\n    for cl in claims:\n        later = sorted(a for x in cl.pop("_ctrls") for a in effects[cl["level"]][x] if a > cl["at_action"])\n        cl["falsified_later"] = bool(later)\n        cl["actions_until_effect"] = (later[0] - cl["at_action"]) if later else None\n    return claims\n\n\nEDGE = 4\n\n\ndef _interior(board_ascii):\n    rows = str(board_ascii or "").split("\\n")\n    return tuple(r[EDGE:-EDGE] for r in rows[EDGE:-EDGE]) if len(rows) > 2 * EDGE else tuple(rows)\n\n\nHUD_FREQ = 0.5\n\n\ndef events_timeline(events_file, hud_freq=HUD_FREQ):\n    """Executed actions from *_events.jsonl: [(action_num, level, control, effect)].\n\n    effect = some board cell changed that is NOT part of the level\'s always-changing background. DUCK 38ay:\n    excluding only the 4 edge cells (still done first) left 73-100% of ALL actions (any control) looking effective, because timers,\n    step bars and animations sit deeper than the edge. So the mask is learned per level: a cell that changes on\n    more than `hud_freq` of that level\'s actions is treated as HUD/animation and ignored. DUCK 38au: the verdict\n    lines analyse() counts miss up to 94% of executed actions; this is the ground-truth alternative."""\n    acts, prev = [], None\n    with open(events_file, encoding="utf-8", errors="replace") as f:\n        for line in f:\n            try:\n                r = json.loads(line)\n            except ValueError:\n                continue\n            board = _interior(r.get("board_ascii"))  # edge HUD strips first; the learned mask catches deeper timers\n            if r.get("type") == "action" and r.get("action_num") is not None and r.get("level") is not None:\n                name = str(r.get("action_name") or r.get("action_display") or "").split("(")[0]\n                changed = set()\n                if prev is not None:\n                    for i, (x, y) in enumerate(zip(prev, board)):\n                        if x != y:\n                            changed.update((i, j) for j, (p_, q_) in enumerate(zip(x, y)) if p_ != q_)\n                acts.append((int(r["action_num"]), int(r["level"]), control_of(name), changed))\n            if r.get("type") in ("action", "initial"):\n                prev = board\n    freq = defaultdict(lambda: defaultdict(int))\n    n_lv = defaultdict(int)\n    for num, lv, c, ch in acts:\n        n_lv[lv] += 1\n        for cell in ch:\n            freq[lv][cell] += 1\n    noisy = {lv: {cell for cell, k in freq[lv].items() if n_lv[lv] >= 4 and k > hud_freq * n_lv[lv]}\n             for lv in n_lv}  # once per level (38bu: was rebuilt per action, quadratic on long passes)\n    return [(num, lv, c, bool(ch - noisy[lv])) for num, lv, c, ch in acts]\n\n\ndef analyse_events(msgs, timeline):\n    """Same claims as analyse(), but tries/no-ops/effects come from the events timeline. The claim\'s position is the\n    latest \'Current state: step S\' seen before it; actions with action_num < S count as before the claim."""\n    level, step = 1, 0\n    claims, seen_claim = [], set()\n    for role, tx, rc in msgs:\n        for m in STATE.finditer(tx):\n            step, level = int(m.group(1)), int(m.group(2))\n        if role == "assistant" and rc:\n            for m in CLAIM.finditer(rc):\n                c = control_of(m.group(1))\n                if (level, c) in seen_claim:\n                    continue\n                seen_claim.add((level, c))\n                ctrls = DIRS if c == "DIRS" else (c,)\n                mine = [t for t in timeline if t[1] == level and t[2] in ctrls]\n                before = [t for t in mine if t[0] < step]\n                after = sorted(t[0] for t in mine if t[0] >= step and t[3])\n                claims.append({"level": level, "control": c, "at_action": step, "tries_before": len(before),\n                               "noops_before": sum(1 for t in before if not t[3]),\n                               "untried": [x for x in ctrls if not any(t[2] == x for t in before)],\n                               "effect_before": any(t[3] for t in before),\n                               "earlier_level": bool(EARLIER.search(m.group(0))),\n                               "sentence": " ".join(m.group(0).split())[:220],\n                               "falsified_later": bool(after),\n                               "actions_until_effect": (after[0] - step + 1) if after else None,\n                               "counted_from": "events"})\n    return claims\n\n\ndef events_file_for(req_path):\n    d, b = os.path.split(req_path)\n    b = b.replace("_requests.jsonl", "_events.jsonl")\n    for c in (os.path.join(d, b), os.path.join(d, "artifacts", b)):\n        if os.path.exists(c):\n            return c\n    return None\n\n\ndef summarize(rows):\n    n = len(rows)\n    if not n:\n        return "CLAIMS_NONE"\n    thin = sum(1 for r in rows if r["tries_before"] <= 2)\n    fals = [r for r in rows if r["falsified_later"]]\n    contra = sum(1 for r in rows if r["effect_before"])\n    earlier = [r for r in rows if r["earlier_level"]]\n    l2 = [r for r in rows if r["level"] >= 2]\n    gaps = sorted(r["actions_until_effect"] for r in fals)\n    med = gaps[len(gaps) // 2] if gaps else None\n    return ("CLAIMS n=%d | <=2 tries behind claim: %d (%.0f%%) | already contradicted at claim time: %d | falsified later "\n            "on same level: %d (%.0f%%), median %s actions later | appeal to an earlier level: %d (of %d claims on L2+)"\n            % (n, thin, 100.0 * thin / n, contra, len(fals), 100.0 * len(fals) / n, med, len(earlier), len(l2)))\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    ap.add_argument("paths", nargs="+")\n    ap.add_argument("--json")\n    a = ap.parse_args(argv)\n    files = []\n    for p in a.paths:\n        files += sorted(glob.glob(os.path.join(p, "*_requests.jsonl"))) if os.path.isdir(p) else [p]\n    rows = []\n    for f in files:\n        game = os.path.basename(f).split("-")[0]\n        ev = events_file_for(f)\n        msgs = chronological(f)\n        for c in (analyse_events(msgs, events_timeline(ev)) if ev else analyse(msgs)):\n            c.update(game=game, file=os.path.basename(f))\n            rows.append(c)\n    for r in rows:\n        print("%-5s L%d %-5s tries=%d noop=%d untried=%s contra=%s later=%s(%s) earlier=%s | %s" % (\n            r["game"], r["level"], r["control"], r["tries_before"], r["noops_before"], ",".join(r["untried"]) or "-",\n            int(r["effect_before"]), int(r["falsified_later"]), r["actions_until_effect"], int(r["earlier_level"]),\n            r["sentence"][:120]))\n    print(summarize(rows))\n    if a.json:\n        json.dump(rows, open(a.json, "w"), indent=1)\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'
open('replay_claims.py', 'w', encoding='utf-8').write(_SRC)
print('wrote replay_claims.py', len(_SRC), 'chars')


In [ ]:
# embedded verbatim from fi939-agent-rebuild/claim_null.py
_SRC = '#!/usr/bin/env python3\n"""FI-939 DUCK 38bu: are the model\'s "control X does nothing" claims informative? (per-action effect perception)\n\n38ay withdrew "83% of inert-control claims are contradicted later" because 73-100% of all actions change the board,\nso "the control changed something later" is near-certain whatever the claim said. It named two things needed to\nrevive the question; this tool supplies both, using only recorded traces (no GPU):\n\n1. A claim classifier. Each negative-control sentence (replay_claims.CLAIM) is put in one class, first match wins:\n   OTHER_LEVEL   appeals to an earlier level ("as in level 1")\n   HYPOTHETICAL  conditional or speculative ("would", "might", "if", "maybe")\n   SITUATIONAL   about one position or one try ("blocked by the wall", "the second RIGHT", "here", "now")\n   REPORT        past-tense report of a try ("RIGHT was a no-op", "click -> no-op", "UP = no-op")\n   DURABLE       present-tense inertness ("UP is a no-op", "MOUSE does nothing"): the belief that matters\n   OTHER         none of the above (tallies, plans, fragments)\n   GENERAL = REPORT + DURABLE + OTHER (the first, coarser cut; kept so earlier readouts stay comparable).\n2. A matched null instead of "contradicted later". For each claim, the same control\'s per-try effect rate on the\n   same level, measured outside the claim\'s window, gives an expected number of effects for the tries the model\n   actually made in the W actions after the claim. Observed / expected (O/E) near 1 means the claim predicted\n   nothing about the control; O/E well below 1 means the claim was informative.\n   Also reported: whether the control\'s most recent try before the claim had an effect (a direct misperception\n   when the class is DURABLE), and the usage ratio U: tries of the control in the window over the tries expected\n   from its share of the level\'s actions outside the window. U < 1 means the claim changed what the model did.\n   Caveat: O/E is conditioned on the model choosing to try; a model that avoids the control except where it\n   expects it to work can show O/E near 1 with an informative claim. Read O/E together with U.\n\nEffect = replay_claims.events_timeline: an interior board change outside the level\'s learned HUD/animation mask.\nIt is still a pixel-level effect, so O/E is an upper bound on "the control really worked"; the --sample listing\nexists so the classes can be read by hand.\n\nUsage: claim_null.py DIR [DIR ...] [--window 20] [--sample N] [--seed 939] [--json OUT]\n"""\nimport argparse, glob, json, os, random, re, sys\nfrom collections import defaultdict\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nfrom replay_claims import (CLAIM, DIRS, EARLIER, STATE, chronological, control_of,  # noqa: E402\n                           events_file_for, events_timeline)\n\nCLASSES = ("GENERAL", "SITUATIONAL", "HYPOTHETICAL", "OTHER_LEVEL")\nSUBCLASSES = ("REPORT", "DURABLE", "OTHER")          # GENERAL split (38bu, after hand-reading the first sample)\nHYPO = re.compile(r"\\b(would|could|might|may|if|whether|unless|maybe|perhaps|possibly|suppose|assume|"\n                  r"let me (try|test|check)|test(ing)? (if|whether))\\b", re.I)\nSITU = re.compile(r"\\b(block(ed|s|ing)?|wall|walls|edge|boundary|border|obstacle|here|now|currently|this time|"\n                  r"again|anymore|any more|at (this|that|the current) (position|point|spot|location|cell)|"\n                  r"from (here|this position)|(first|second|third|fourth|fifth|last|\\d+(st|nd|rd|th)) "\n                  r"(up|down|left|right|click|press|move|try|attempt|one|time)|in this state|in that state|"\n                  r"already at|at the (top|bottom|left|right))\\b", re.I)\n\n\nREPORT = re.compile(r"\\b(was|were|did|got|gave|returned|produced|resulted|happened|tried|tested|pressed|clicked)\\b"\n                    r"|->|\\u2192|=", re.I)\nDURABLE = re.compile(r"\\b(is|are|does|do|has|have|seems?|appears?)\\b[^.]{0,40}?\\b(inert|no-?ops?|nothing|"\n                     r"no effect|useless|non-functional)\\b", re.I)\n\n\ndef subclass(sentence):\n    """Split of GENERAL; other classes map to themselves."""\n    c = classify(sentence)\n    if c != "GENERAL":\n        return c\n    if REPORT.search(sentence):\n        return "REPORT"\n    if DURABLE.search(sentence):\n        return "DURABLE"\n    return "OTHER"\n\n\ndef classify(sentence):\n    if EARLIER.search(sentence):\n        return "OTHER_LEVEL"\n    if HYPO.search(sentence):\n        return "HYPOTHETICAL"\n    if SITU.search(sentence):\n        return "SITUATIONAL"\n    return "GENERAL"\n\n\ndef claims_with_null(msgs, timeline, window=20):\n    """One row per (level, control, class): the first such sentence. Positions use \'Current state: step S\'."""\n    by_lc = defaultdict(list)                      # (level, control) -> [(num, effect)]\n    lv_nums = defaultdict(list)                    # level -> [action num]\n    for num, lv, c, eff in timeline:\n        by_lc[(lv, c)].append((num, eff))\n        lv_nums[lv].append(num)\n    level, step = 1, 0\n    rows, seen = [], set()\n    for role, tx, rc in msgs:\n        for m in STATE.finditer(tx):\n            step, level = int(m.group(1)), int(m.group(2))\n        if role != "assistant" or not rc:\n            continue\n        for m in CLAIM.finditer(rc):\n            c = control_of(m.group(1))\n            sent = " ".join(m.group(0).split())\n            cls = classify(sent)\n            if (level, c, cls) in seen:\n                continue\n            seen.add((level, c, cls))\n            ctrls = DIRS if c == "DIRS" else (c,)\n            tries = sorted(t for x in ctrls for t in by_lc.get((level, x), []))\n            before = [t for t in tries if t[0] < step]\n            post = [t for t in tries if step <= t[0] < step + window]\n            outside = [t for t in tries if not (step <= t[0] < step + window)]\n            rate = (sum(1 for t in outside if t[1]) / len(outside)) if outside else None\n            win_n = sum(1 for n in lv_nums.get(level, []) if step <= n < step + window)\n            out_n = len(lv_nums.get(level, [])) - win_n\n            share = (len(outside) / out_n) if out_n else None\n            rows.append({"level": level, "control": c, "cls": cls, "sub": subclass(sent), "step": step,\n                         "sentence": sent[:220],\n                         "win_actions": win_n, "use_share": share,\n                         "expected_tries": (share * win_n) if share is not None else None,\n                         "tries_before": len(before),\n                         "last_try_effect": (before[-1][1] if before else None),\n                         "post_tries": len(post), "post_effects": sum(1 for t in post if t[1]),\n                         "base_rate": rate,\n                         "expected": (rate * len(post)) if rate is not None else None})\n    return rows\n\n\ndef oe(rows, rng=None, boots=0):\n    """Observed/expected effects over rows that have post tries and a base rate; optional bootstrap 95% CI."""\n    use = [r for r in rows if r["post_tries"] and r["expected"] is not None]\n    o = sum(r["post_effects"] for r in use)\n    e = sum(r["expected"] for r in use)\n    out = {"n": len(use), "obs": o, "exp": round(e, 2), "oe": (o / e if e else None), "ci": None}\n    if boots and use and rng is not None:\n        vals = []\n        for _ in range(boots):\n            s = [use[rng.randrange(len(use))] for _ in use]\n            se = sum(r["expected"] for r in s)\n            if se:\n                vals.append(sum(r["post_effects"] for r in s) / se)\n        vals.sort()\n        if vals:\n            out["ci"] = (vals[int(0.025 * len(vals))], vals[int(0.975 * len(vals)) - 1])\n    return out\n\n\ndef usage(rows):\n    """U = observed tries in the window / tries expected from the control\'s share outside it."""\n    use = [r for r in rows if r.get("expected_tries") is not None and r.get("win_actions")]\n    o = sum(r["post_tries"] for r in use)\n    e = sum(r["expected_tries"] for r in use)\n    return {"n": len(use), "obs": o, "exp": round(e, 2), "u": (o / e if e else None)}\n\n\ndef summarize(rows, rng=None, boots=0):\n    lines = []\n    for cls in CLASSES + SUBCLASSES + ("ALL",):\n        if cls == "ALL":\n            rs = rows\n        elif cls in SUBCLASSES:\n            rs = [r for r in rows if r.get("sub") == cls]\n        else:\n            rs = [r for r in rows if r["cls"] == cls]\n        if not rs:\n            lines.append("%-12s n=0" % cls)\n            continue\n        k = oe(rs, rng, boots)\n        lt = [r for r in rs if r["last_try_effect"] is not None]\n        mis = sum(1 for r in lt if r["last_try_effect"])\n        ci = (" CI95 %.2f-%.2f" % k["ci"]) if k["ci"] else ""\n        u = usage(rs)\n        ci += (" | usage U=%.2f (tries %d vs %.1f expected)" % (u["u"], u["obs"], u["exp"])) if u["u"] is not None else ""\n        lines.append(("  " if cls in SUBCLASSES else "") + "%-12s n=%d | last try before claim HAD an effect: %d/%d (%s) | post-claim O/E=%s (obs %d, "\n                     "exp %.1f, %d claims with tries)%s"\n                     % (cls, len(rs), mis, len(lt), ("%.0f%%" % (100.0 * mis / len(lt))) if lt else "-",\n                        ("%.2f" % k["oe"]) if k["oe"] is not None else "-", k["obs"], k["exp"], k["n"], ci))\n    return "\\n".join(lines)\n\n\ndef run(dirs, window=20):\n    rows = []\n    for d in dirs:\n        files = sorted(glob.glob(os.path.join(d, "*_requests.jsonl"))) if os.path.isdir(d) else [d]\n        for f in files:\n            ev = events_file_for(f)\n            if not ev:\n                continue\n            for r in claims_with_null(chronological(f), events_timeline(ev), window):\n                r.update(game=os.path.basename(f).split("-")[0], file=os.path.basename(f), arm=os.path.basename(\n                    os.path.dirname(os.path.abspath(f))))\n                rows.append(r)\n    return rows\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    ap.add_argument("paths", nargs="+")\n    ap.add_argument("--window", type=int, default=20)\n    ap.add_argument("--sample", type=int, default=0, help="print N random claims per class for hand reading")\n    ap.add_argument("--seed", type=int, default=939)\n    ap.add_argument("--boots", type=int, default=1000)\n    ap.add_argument("--json")\n    a = ap.parse_args(argv)\n    rows = run(a.paths, a.window)\n    rng = random.Random(a.seed)\n    print("CLAIM_NULL window=%d actions, %d claim rows from %s" % (a.window, len(rows), ", ".join(a.paths)))\n    print(summarize(rows, rng, a.boots))\n    if a.sample:\n        for cls in SUBCLASSES + CLASSES[1:]:\n            rs = [r for r in rows if r.get("sub") == cls]\n            for r in rng.sample(rs, min(a.sample, len(rs))):\n                print("SAMPLE %-12s %s L%d %-5s last=%s post=%d/%d base=%s | %s" % (\n                    cls, r["game"], r["level"], r["control"], r["last_try_effect"], r["post_effects"],\n                    r["post_tries"], ("%.2f" % r["base_rate"]) if r["base_rate"] is not None else "-",\n                    r["sentence"][:160]))\n    if a.json:\n        with open(a.json, "w") as f:\n            json.dump(rows, f, indent=1)\n    return 0 if rows else 3\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'
open('claim_null.py', 'w', encoding='utf-8').write(_SRC)
print('wrote claim_null.py', len(_SRC), 'chars')


In [ ]:
# embedded verbatim from fi939-agent-rebuild/replay_tests.py
_SRC = '#!/usr/bin/env python3\n"""FI-939 DUCK 38bw: does the model\'s hypothesis TESTING target what it does not know? (replay, no GPU)\n\nHypothesis generate/test/revise is one of this project\'s research lines (set 2026-10-07). 38bu showed the model\'s negative\ncontrol statements carry no information. This asks the matching question about its experiments: when the\nreasoning announces a test of a control ("let me test whether UP moves the block", "I\'ll check if clicking ..."),\n(1) is that control actually executed soon after, and (2) was the test aimed at an uncertain control?\n\nUncertainty is measured from the environment log only. Before each executed action, the control\'s effect rate so\nfar on this level is p = (effects + 1) / (tries + 2) (Laplace), and the action\'s expected information is the binary\nentropy H(p) in bits: 1.0 for a control never tried or 50/50, near 0 for one that always (or never) worked.\n\'Effect\' is replay_claims.events_timeline\'s (interior change outside the level\'s learned HUD/animation mask).\n\nReported per arm:\n  announcements  test sentences naming a control (deduped per level, step, control)\n  executed@K     share whose control is executed within K actions on the same level after the announcement\n  H(test)        mean H(p) at the first such execution\n  H(all)         mean H(p) over every executed action (the null: what a typical action is expected to reveal)\n  surprise       mean -log2 P(observed outcome) at the test execution vs over all actions\nA test that targets uncertainty has H(test) > H(all). A test that only re-confirms has H(test) near 0.\n\nUsage: replay_tests.py DIR [DIR ...] [--k 5] [--json OUT] [--per-game]\n"""\nimport argparse, glob, json, math, os, re, sys\nfrom collections import defaultdict\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nfrom replay_claims import CTRL, DIRS, STATE, chronological, control_of, events_file_for, events_timeline  # noqa: E402\n\nTEST = re.compile(r"[^.\\n]*\\b(let me|let\'s|lets|i\'ll|i will|i need to|i should|need to|going to|now)\\s+"\n                  r"(test|check|verify|probe|see|try)\\b[^.\\n]*", re.I)\nCTRL_RE = re.compile(r"\\b" + CTRL + r"\\b", re.I)\n\n\ndef h2(p):\n    return 0.0 if p <= 0 or p >= 1 else -(p * math.log2(p) + (1 - p) * math.log2(1 - p))\n\n\ndef annotate(timeline):\n    """[(num, level, control, effect, H, surprise, tries_before)]; p from the control\'s record on the level BEFORE it."""\n    rec = defaultdict(lambda: [0, 0])\n    out = []\n    for num, lv, c, eff in timeline:\n        t = rec[(lv, c)]\n        p = (t[1] + 1.0) / (t[0] + 2.0)\n        out.append((num, lv, c, eff, h2(p), -math.log2(p if eff else 1 - p), t[0]))\n        t[0] += 1\n        t[1] += 1 if eff else 0\n    return out\n\n\ndef announcements(msgs):\n    level, step, seen, out = 1, 0, set(), []\n    for role, tx, rc in msgs:\n        for m in STATE.finditer(tx):\n            step, level = int(m.group(1)), int(m.group(2))\n        if role != "assistant" or not rc:\n            continue\n        for m in TEST.finditer(rc):\n            sent = " ".join(m.group(0).split())\n            for cm in CTRL_RE.finditer(sent):\n                c = control_of(cm.group(1))\n                if (level, step, c) in seen:\n                    continue\n                seen.add((level, step, c))\n                out.append({"level": level, "step": step, "control": c, "sentence": sent[:200]})\n    return out\n\n\ndef analyse(msgs, timeline, k=5):\n    ann = annotate(timeline)\n    rows = []\n    for a in announcements(msgs):\n        ctrls = DIRS if a["control"] == "DIRS" else (a["control"],)\n        lvl = [x for x in ann if x[1] == a["level"] and x[0] >= a["step"]][:k]\n        hit = next((x for x in lvl if x[2] in ctrls), None)\n        a.update(executed=hit is not None, H=(hit[4] if hit else None), surprise=(hit[5] if hit else None),\n                 effect=(hit[3] if hit else None), tries_before=(hit[6] if hit else None))\n        rows.append(a)\n    return rows, ann\n\n\nSTRATA = ((0, 0), (1, 2), (3, 5), (6, 10 ** 9))\n\n\ndef strata(rows, ann):\n    """H and surprise of test executions vs all actions, within bands of prior tries of that control on the level.\n    Without this, tests look well aimed just because they come early (a first try is always H=1)."""\n    out = []\n    for lo, hi in STRATA:\n        ex = [r for r in rows if r["executed"] and lo <= r["tries_before"] <= hi]\n        al = [x for x in ann if lo <= x[6] <= hi]\n        mean = lambda xs: sum(xs) / len(xs) if xs else float("nan")\n        out.append("  prior tries %s: tests n=%d H=%.2f surprise=%.2f | all n=%d H=%.2f surprise=%.2f" % (\n            ("%d" % lo) if lo == hi else ("%d+" % lo if hi > 10 ** 6 else "%d-%d" % (lo, hi)),\n            len(ex), mean([r["H"] for r in ex]), mean([r["surprise"] for r in ex]),\n            len(al), mean([x[4] for x in al]), mean([x[5] for x in al])))\n    return "\\n".join(out)\n\n\ndef summarize(rows, ann, label="ALL"):\n    n = len(rows)\n    ex = [r for r in rows if r["executed"]]\n    ha = sum(x[4] for x in ann) / len(ann) if ann else float("nan")\n    sa = sum(x[5] for x in ann) / len(ann) if ann else float("nan")\n    ht = sum(r["H"] for r in ex) / len(ex) if ex else float("nan")\n    st = sum(r["surprise"] for r in ex) / len(ex) if ex else float("nan")\n    low = sum(1 for r in ex if r["H"] < 0.5)\n    return ("%-6s announcements=%d executed=%d (%s) | H(test)=%.2f H(all)=%.2f bits | surprise test=%.2f all=%.2f | "\n            "tests on near-certain controls (H<0.5): %d (%s) | actions=%d"\n            % (label, n, len(ex), ("%.0f%%" % (100.0 * len(ex) / n)) if n else "-", ht, ha, st, sa, low,\n               ("%.0f%%" % (100.0 * low / len(ex))) if ex else "-", len(ann)))\n\n\ndef run(paths, k=5):\n    rows, ann_all, per_game = [], [], defaultdict(lambda: ([], []))\n    for p in paths:\n        files = sorted(glob.glob(os.path.join(p, "*_requests.jsonl"))) if os.path.isdir(p) else [p]\n        for f in files:\n            ev = events_file_for(f)\n            if not ev:\n                continue\n            r, a = analyse(chronological(f), events_timeline(ev), k)\n            g = os.path.basename(f).split("-")[0]\n            for x in r:\n                x.update(game=g, file=os.path.basename(f))\n            rows += r\n            ann_all += a\n            per_game[g][0].extend(r)\n            per_game[g][1].extend(a)\n    return rows, ann_all, per_game\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    ap.add_argument("paths", nargs="+")\n    ap.add_argument("--k", type=int, default=5)\n    ap.add_argument("--json")\n    ap.add_argument("--per-game", action="store_true")\n    a = ap.parse_args(argv)\n    rows, ann, per_game = run(a.paths, a.k)\n    print("REPLAY_TESTS k=%d from %s" % (a.k, ", ".join(a.paths)))\n    print(summarize(rows, ann))\n    print(strata(rows, ann))\n    if a.per_game:\n        for g in sorted(per_game):\n            print(summarize(per_game[g][0], per_game[g][1], g))\n    if a.json:\n        with open(a.json, "w") as f:\n            json.dump(rows, f, indent=1)\n    return 0 if ann else 3\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'
open('replay_tests.py', 'w', encoding='utf-8').write(_SRC)
print('wrote replay_tests.py', len(_SRC), 'chars')


In [ ]:
# embedded verbatim from fi939-agent-rebuild/replay_revise.py
_SRC = '#!/usr/bin/env python3\n"""FI-939 DUCK 38bx: the REVISE step - after the environment contradicts "X does nothing", does the model update?\n\n38bu: inert-control statements carry no information. 38bw: the experiments behind them are well aimed. So the loss\nshould sit in revision. This measures it directly from recorded traces (no GPU).\n\nFor each pass and each (level, control) with a negative statement (claim_null classes; default: DURABLE and REPORT,\ni.e. GENERAL beliefs), find the first CONTRADICTION: an action of that control on the same level, at or after the\nstatement, whose board effect is real (replay_claims.events_timeline: interior change outside the learned HUD mask).\nThen read the model\'s later reasoning on that level, in order, for the first sentence that names the control and is\neither\n  POSITIVE  the control moved / changed / worked / rotated / selected / did something, or\n  NEGATIVE  another inert statement (replay_claims.CLAIM).\nOutcome per contradiction: REVISED (positive first), REASSERTED (negative first) or SILENT (neither before the level\nends). Also: assistant turns from contradiction to the first positive mention.\n\nPixel-level effect is an upper bound on "the control did what the model cared about", so REASSERTED can include\nfair calls (an effect the model rightly treats as irrelevant). The --sample listing exists for hand reading.\n\nUsage: replay_revise.py DIR [DIR ...] [--classes DURABLE,REPORT] [--sample N] [--seed 939] [--json OUT]\n"""\nimport argparse, glob, json, os, random, re, sys\nfrom collections import Counter, defaultdict\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nfrom replay_claims import CLAIM, CTRL, DIRS, STATE, chronological, control_of, events_file_for, events_timeline  # noqa: E402\nfrom claim_null import subclass  # noqa: E402\n\nPOS = re.compile(r"[^.\\n]*\\b" + CTRL + r"\\b[^.\\n]{0,60}?\\b(moves?|moved|works?|worked|changes?|changed|rotates?|"\n                 r"rotated|shifts?|shifted|selects?|selected|toggles?|toggled|did something|does something|"\n                 r"has an effect|had an effect|is effective|responds?|responded|pushes|pushed|activates?|activated)\\b"\n                 r"[^.\\n]*", re.I)\n\n\ndef mentions(sentence_text):\n    """[(kind, control, sentence)] for positive and negative statements in a reasoning block, in text order."""\n    out = []\n    for kind, rx in (("POS", POS), ("NEG", CLAIM)):\n        for m in rx.finditer(sentence_text):\n            out.append((m.start(), kind, control_of(m.group(1)), " ".join(m.group(0).split())))\n    out.sort()\n    res = []\n    for _, kind, c, s in out:\n        if kind == "POS" and CLAIM.search(s):      # "UP moved nothing" style: negative wins\n            kind = "NEG"\n        res.append((kind, c, s))\n    return res\n\n\ndef _match(c, ctrls):\n    return c in ctrls or (c == "DIRS" and set(ctrls) <= set(DIRS)) or (c in DIRS and "DIRS" in ctrls)\n\n\ndef analyse(msgs, timeline, classes=("DURABLE", "REPORT")):\n    # message index -> (level, step) as of that message; reasoning blocks of assistant turns\n    level, step = 1, 0\n    turns = []                                       # (idx, level, step, reasoning)\n    for i, (role, tx, rc) in enumerate(msgs):\n        for m in STATE.finditer(tx):\n            step, level = int(m.group(1)), int(m.group(2))\n        if role == "assistant" and rc:\n            turns.append((i, level, step, rc))\n    rows, seen = [], set()\n    for ti, (i, lv, st, rc) in enumerate(turns):\n        for kind, c, sent in mentions(rc):\n            if kind != "NEG" or subclass(sent) not in classes or (lv, c) in seen:\n                continue\n            seen.add((lv, c))\n            ctrls = DIRS if c == "DIRS" else (c,)\n            contra = next((t for t in timeline if t[1] == lv and t[2] in ctrls and t[0] >= st and t[3]), None)\n            row = {"level": lv, "control": c, "step": st, "sentence": sent[:200], "contradicted": contra is not None,\n                   "contra_step": contra[0] if contra else None, "outcome": None, "turns_to": None, "reply": None}\n            if contra:\n                later = [(tj, x) for tj, x in enumerate(turns) if tj > ti and x[2] > contra[0]]\n                n = 0\n                for tj, (_, lv2, st2, rc2) in later:\n                    if lv2 != lv:\n                        break\n                    n += 1\n                    hit = next(((k, s) for k, c2, s in mentions(rc2) if _match(c2, ctrls)), None)\n                    if hit:\n                        row.update(outcome="REVISED" if hit[0] == "POS" else "REASSERTED", turns_to=n,\n                                   reply=hit[1][:200])\n                        break\n                if row["outcome"] is None:\n                    row["outcome"] = "SILENT"\n            rows.append(row)\n    return rows\n\n\ndef summarize(rows):\n    c = [r for r in rows if r["contradicted"]]\n    k = Counter(r["outcome"] for r in c)\n    rev = sorted(r["turns_to"] for r in c if r["outcome"] == "REVISED")\n    med = rev[len(rev) // 2] if rev else None\n    n = len(c)\n    pct = lambda x: ("%.0f%%" % (100.0 * x / n)) if n else "-"\n    return ("REVISE statements=%d contradicted on the same level=%d | REVISED %d (%s), REASSERTED %d (%s), SILENT %d (%s) "\n            "| median turns to revision %s" % (len(rows), n, k["REVISED"], pct(k["REVISED"]), k["REASSERTED"],\n                                                pct(k["REASSERTED"]), k["SILENT"], pct(k["SILENT"]), med))\n\n\ndef run(paths, classes):\n    rows = []\n    for p in paths:\n        files = sorted(glob.glob(os.path.join(p, "*_requests.jsonl"))) if os.path.isdir(p) else [p]\n        for f in files:\n            ev = events_file_for(f)\n            if not ev:\n                continue\n            for r in analyse(chronological(f), events_timeline(ev), classes):\n                r.update(game=os.path.basename(f).split("-")[0], file=os.path.basename(f))\n                rows.append(r)\n    return rows\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    ap.add_argument("paths", nargs="+")\n    ap.add_argument("--classes", default="DURABLE,REPORT")\n    ap.add_argument("--sample", type=int, default=0)\n    ap.add_argument("--seed", type=int, default=939)\n    ap.add_argument("--json")\n    a = ap.parse_args(argv)\n    classes = tuple(x.strip() for x in a.classes.split(",") if x.strip())\n    rows = run(a.paths, classes)\n    print("REPLAY_REVISE classes=%s from %s" % (",".join(classes), ", ".join(a.paths)))\n    print(summarize(rows))\n    for cls in classes:\n        print("  %-8s %s" % (cls, summarize([r for r in rows if subclass(r["sentence"]) == cls])))\n    if a.sample:\n        rng = random.Random(a.seed)\n        for oc in ("REVISED", "REASSERTED", "SILENT"):\n            rs = [r for r in rows if r["outcome"] == oc]\n            for r in rng.sample(rs, min(a.sample, len(rs))):\n                print("SAMPLE %-10s %s L%d %-5s | %s || %s" % (oc, r["game"], r["level"], r["control"],\n                                                             r["sentence"][:110], (r["reply"] or "")[:110]))\n    if a.json:\n        with open(a.json, "w") as f:\n            json.dump(rows, f, indent=1)\n    return 0 if rows else 3\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'
open('replay_revise.py', 'w', encoding='utf-8').write(_SRC)
print('wrote replay_revise.py', len(_SRC), 'chars')


In [ ]:
# embedded verbatim from fi939-agent-rebuild/ab_metrics.py
_SRC = '#!/usr/bin/env python3\n"""FI-939 tier-2 A/B metrics extractor.\n\nParses a tier-2 run directory (produced by tier2_ab_run.sh / inference-taaf-run inline) and\nprints per-game behavioral signals for baseline-vs-patched comparison:\n  analyses, actions, board-changes, game_over/resets, levels_completed, plus note-quality\n  from the transcript (Progress-ledger present, unique Confirmed-mechanics lines).\n\nUsage: ~/duck-venv/bin/python ab_metrics.py <run_dir> [run_dir2 ...]\nA run_dir is ~/duck_ab/<name> (the parent; the timestamped subdir is found automatically).\n"""\nimport sys, os, json, glob, re\n\ndef _bool(v): return bool(v) and v not in (0, "0", "false", "False", "")\n\ndef game_from_events(path):\n    n_analysis = n_action = n_board = n_go = n_level = n_run = n_levels = 0\n    for line in open(path, encoding="utf-8", errors="replace"):\n        line = line.strip()\n        if not line:\n            continue\n        try:\n            e = json.loads(line)\n        except Exception:\n            continue\n        t = e.get("type") or e.get("event")\n        if t == "analysis":\n            n_analysis += 1\n        elif t == "action":\n            n_action += 1\n            # result fields may be nested under \'result\'/\'last_action_result\' or top-level\n            r = e.get("result") or e.get("last_action_result") or e\n            if _bool(r.get("board_changed")): n_board += 1\n            if _bool(r.get("game_over")):      n_go += 1\n            if _bool(r.get("level_completed")):n_level += 1\n            if _bool(r.get("run_complete")):   n_run += 1\n            # 2026-10-29: the action that clears a game\'s LAST level carries run_complete, not level_completed\n            # (measured: 32/32 won games in hf-base). \'levels\' counts it, matching the official scorer, which\n            # counts score increments. levels_completed keeps its old meaning for existing readouts.\n            if _bool(r.get("level_completed")) or _bool(r.get("run_complete")): n_levels += 1\n    return dict(analyses=n_analysis, actions=n_action, board_changes=n_board,\n                resets_game_over=n_go, levels_completed=n_level, run_complete=n_run, levels=n_levels)\n\ndef note_quality(transcript):\n    if not transcript or not os.path.exists(transcript):\n        return dict(progress_ledger=False, confirmed_unique=0)\n    txt = open(transcript, encoding="utf-8", errors="replace").read()\n    ledger = "Progress ledger" in txt\n    # unique confirmed-mechanics bullet lines the model emitted (rough flip-flop/accumulation proxy)\n    conf = set()\n    for m in re.finditer(r"[Cc]onfirmed mechanics:\\s*(.+)", txt):\n        frag = m.group(1).strip()\n        if frag and "PROVEN" not in frag and "each with" not in frag:  # skip the prompt template line\n            conf.add(frag[:80].lower())\n    return dict(progress_ledger=ledger, confirmed_unique=len(conf))\n\ndef resolve(run_dir):\n    run_dir = os.path.expanduser(run_dir)\n    subs = sorted(glob.glob(os.path.join(run_dir, "*/")))\n    return subs[-1].rstrip("/") if subs else run_dir\n\ndef main(argv):\n    if len(argv) < 2:\n        print(__doc__); return 2\n    for rd in argv[1:]:\n        d = resolve(rd)\n        print(f"\\n=== RUN: {d} ===")\n        evs = sorted(glob.glob(os.path.join(d, "artifacts", "*events.jsonl")))\n        if not evs:\n            print("  (no events artifacts)"); continue\n        for ev in evs:\n            base = os.path.basename(ev).replace("_events.jsonl", "")\n            g = game_from_events(ev)\n            tr = glob.glob(os.path.join(d, "transcripts", base + "*.txt"))\n            nq = note_quality(tr[0] if tr else None)\n            print(f"  {base}: analyses={g[\'analyses\']} actions={g[\'actions\']} "\n                  f"board_changes={g[\'board_changes\']} resets(game_over)={g[\'resets_game_over\']} "\n                  f"levels={g[\'levels_completed\']} | ledger={nq[\'progress_ledger\']} "\n                  f"confirmed_unique={nq[\'confirmed_unique\']}")\n    return 0\n\nif __name__ == "__main__":\n    sys.exit(main(sys.argv))\n'
open('ab_metrics.py', 'w', encoding='utf-8').write(_SRC)
print('wrote ab_metrics.py', len(_SRC), 'chars')


In [ ]:
# embedded verbatim from fi939-agent-rebuild/fi939_gate.py
_SRC = '#!/usr/bin/env python3\n"""FI-939: apply the pre-registered LoRA slot rule to an HF 25-game screen, mechanically.\n\nRule (pre-registered before the G6 screen ran): a candidate adapter earns a Kaggle slot only if\n  (1) held-out games: sum of per-game mean levels is NOT below base (or, with --min-heldout-gain-pct X,\n      at least X percent above base), AND\n  (2) all games: total is not significantly worse than base, tested with a one-sided paired sign-flip\n      permutation test over per-game mean differences (p_worse < alpha => significantly worse).\nPer-game means are used because pass counts can differ per game (hard stops, olen gate).\nLevels come from ab_metrics.game_from_events, the same completion rule the fleet\'s A/B readouts use.\n\nusage: fi939_gate.py --base hf-base --cand hf-lora-g6 --heldout bp35,cd82,... [--traces DIR] [--json]\n--base/--cand also take comma-separated run dirs (e.g. hf-lora-g12,hf-lora-g12r): their passes are POOLED per game\nbefore the per-game means, so replicate screens are read as one larger sample (2026-10-22, G12 replicate).\nexit 0 = PASS, 1 = FAIL, 2 = cannot evaluate (missing games / data).\n"""\nimport argparse, glob, json, os, random, re, sys\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, HERE)\nEV = re.compile(r"([a-z0-9]{4})-[0-9a-f]{8}_p(\\d+)_events\\.jsonl$")\n\n\ndef per_game(run_dir, levels_fn=None):\n    if levels_fn is None:\n        from ab_metrics import game_from_events\n        # \'levels\' includes a won game\'s last level (levels_completed misses it); see ab_metrics, 2026-10-29\n        levels_fn = lambda p: game_from_events(p)["levels"]\n    g = {}\n    for p in sorted(glob.glob(os.path.join(run_dir, "artifacts", "*_events.jsonl"))):\n        m = EV.match(os.path.basename(p))\n        if m:\n            g.setdefault(m.group(1), []).append(levels_fn(p))\n    return g\n\n\ndef pooled_per_game(traces, names, per_game_fn=None):\n    """Merge the per-game pass lists of several run dirs (comma-separated names) into one sample."""\n    per_game_fn = per_game_fn or per_game\n    out = {}\n    for n in [x.strip() for x in names.split(",") if x.strip()]:\n        for g, v in per_game_fn(os.path.join(traces, n)).items():\n            out.setdefault(g, []).extend(v)\n    return out\n\n\ndef sign_flip_p_worse(diffs, n_perm=200000, seed=939):\n    """One-sided p that mean(cand-base) is this low or lower by chance (H0: symmetric around 0)."""\n    obs = sum(diffs)\n    nz = [d for d in diffs if d != 0]\n    n = len(nz)\n    if n == 0:\n        return 1.0\n    if n <= 20:  # exact\n        cnt = 0\n        for mask in range(1 << n):\n            s = sum(-d if (mask >> i) & 1 else d for i, d in enumerate(nz))\n            cnt += s <= obs + 1e-12\n        return cnt / (1 << n)\n    rng = random.Random(seed)\n    cnt = 0\n    for _ in range(n_perm):\n        s = sum(d if rng.random() < 0.5 else -d for d in nz)\n        cnt += s <= obs + 1e-12\n    return (cnt + 1) / (n_perm + 1)\n\n\ndef evaluate(base, cand, heldout, alpha=0.05, min_games=20, min_heldout_gain_pct=0.0):\n    games = sorted(set(base) & set(cand))\n    missing_held = [k for k in heldout if k not in games]\n    out = {"games": len(games), "heldout": list(heldout), "missing_heldout": missing_held, "alpha": alpha,\n           "min_heldout_gain_pct": min_heldout_gain_pct}\n    if len(games) < min_games or missing_held:\n        out["verdict"] = "CANNOT_EVALUATE"\n        return out\n    mb = {k: sum(base[k]) / len(base[k]) for k in games}\n    mc = {k: sum(cand[k]) / len(cand[k]) for k in games}\n    tb, tc = sum(mb.values()), sum(mc.values())\n    hb, hc = sum(mb[k] for k in heldout), sum(mc[k] for k in heldout)\n    diffs = [mc[k] - mb[k] for k in games]\n    p = sign_flip_p_worse(diffs)\n    out.update({\n        "total_base": round(tb, 3), "total_cand": round(tc, 3), "total_pct": round(100 * (tc - tb) / tb, 2) if tb else None,\n        "heldout_base": round(hb, 3), "heldout_cand": round(hc, 3),\n        "heldout_pct": round(100 * (hc - hb) / hb, 2) if hb else None,\n        "up_same_down": [sum(mc[k] > mb[k] for k in games), sum(mc[k] == mb[k] for k in games),\n                         sum(mc[k] < mb[k] for k in games)],\n        "p_worse": round(p, 4),\n        "rule_heldout_not_worse": (hc >= hb) if min_heldout_gain_pct <= 0 else\n                                  (hb > 0 and 100 * (hc - hb) / hb >= min_heldout_gain_pct),\n        "rule_total_not_sig_worse": p >= alpha,\n    })\n    out["verdict"] = "PASS" if out["rule_heldout_not_worse"] and out["rule_total_not_sig_worse"] else "FAIL"\n    return out\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--base", required=True)\n    ap.add_argument("--cand", required=True)\n    ap.add_argument("--heldout", required=True)\n    ap.add_argument("--traces", default=os.path.expanduser(r"~\\m2\\traces") if os.name == "nt" else os.path.expanduser("~/m2/traces"))\n    ap.add_argument("--alpha", type=float, default=0.05)\n    ap.add_argument("--min-heldout-gain-pct", type=float, default=0.0,\n                    help="held-out leg requires cand >= base by this many percent (0 = not worse). Added 2026-10-20 after "\n                         "G5 scored 24.12 on Kaggle vs base mean 26.77 while the screen total said +0.3%%")\n    ap.add_argument("--json", action="store_true")\n    a = ap.parse_args()\n    held = [h.strip() for h in a.heldout.split(",") if h.strip()]\n    r = evaluate(pooled_per_game(a.traces, a.base), pooled_per_game(a.traces, a.cand), held, a.alpha,\n                 min_heldout_gain_pct=a.min_heldout_gain_pct)\n    r.update({"base": a.base, "cand": a.cand})\n    if a.json:\n        print(json.dumps(r))\n    else:\n        for k, v in r.items():\n            print("%-26s %s" % (k, v))\n    sys.exit({"PASS": 0, "FAIL": 1}.get(r["verdict"], 2))\n\n\nif __name__ == "__main__":\n    main()\n'
open('fi939_gate.py', 'w', encoding='utf-8').write(_SRC)
print('wrote fi939_gate.py', len(_SRC), 'chars')


In [ ]:
# embedded verbatim from fi939-agent-rebuild/loss_decompose.py
_SRC = '#!/usr/bin/env python3\n"""FI-939 DUCK 38cd: where does the official score go - unfinished levels or inefficiency on finished ones?\n\nThe official game score weights level L by L and scores each completed level min((human/agent actions)^2, 1.15),\ncapped at the completed levels\' weight share. So 1 - score splits exactly into:\n  unfinished    1 - compl            (compl = the score if every completed level scored 1.0)\n  inefficiency  compl - official     (completed levels scored below 1.0 because of extra actions)\nEfficiency arms can only recover the second part; level-clearing arms the first.\n\nThe tool also reports where runs get stuck: the first uncleared level per game-pass. Because later levels carry\nmore weight, getting past level k unlocks the weight of every level after it.\n\nUses duck_score_estimate\'s scoring (level_actions, level_score, game_score) on every pass, not just p0.\n\nUsage: loss_decompose.py RUN_DIR [RUN_DIR ...] [--env-dir DIR] [--per-game]\n"""\nimport argparse, glob, json, os, re, sys\nfrom collections import Counter, defaultdict\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport duck_score_estimate as D  # noqa: E402\n\nNAME = re.compile(r"^(?P<g>[a-z0-9]+(?:-[0-9a-f]+)?)_p(?P<p>\\d+)_events\\.jsonl$")\n\n\ndef score_pass(path, baselines):\n    m = NAME.match(os.path.basename(path))\n    if not m:\n        return None\n    game = m.group("g")\n    base = baselines.get(game) or baselines.get(game.split("-")[0])\n    if not base:\n        return None\n    events = []\n    with open(path, encoding="utf-8", errors="replace") as f:\n        for line in f:\n            try:\n                events.append(json.loads(line))\n            except ValueError:\n                pass\n    lv = D.level_actions(events)\n    cum = {L: D.level_score(base[L - 1], c) for L, c, _ in lv if L <= len(base)}\n    full = {L: 1.0 for L, _, _ in lv if L <= len(base)}\n    done = sorted(cum)\n    stuck = next((L for L in range(1, len(base) + 1) if L not in cum), None)   # None = all levels cleared\n    return {"game": game.split("-")[0], "pass": int(m.group("p")), "n_levels": len(base),\n            "official": D.game_score(base, cum), "compl": D.game_score(base, full), "levels": len(done),\n            "stuck": stuck}\n\n\ndef summarize(rows):\n    """Mean over passes within a game, then over games (the competition averages games)."""\n    by = defaultdict(list)\n    for r in rows:\n        by[r["game"]].append(r)\n    g_off = {g: sum(r["official"] for r in v) / len(v) for g, v in by.items()}\n    g_cmp = {g: sum(r["compl"] for r in v) / len(v) for g, v in by.items()}\n    n = len(by)\n    off = sum(g_off.values()) / n if n else 0.0\n    cmp_ = sum(g_cmp.values()) / n if n else 0.0\n    stuck = Counter(("all cleared" if r["stuck"] is None else ("L1" if r["stuck"] == 1 else "L2" if r["stuck"] == 2\n                     else "L3+")) for r in rows)\n    return {"games": n, "passes": len(rows), "official": off, "compl": cmp_, "unfinished": 1 - cmp_,\n            "inefficiency": cmp_ - off, "stuck": stuck, "per_game": {g: (g_off[g], g_cmp[g], len(by[g])) for g in by}}\n\n\ndef render(s, label):\n    lost = 1 - s["official"]\n    st = s["stuck"]\n    tot = sum(st.values()) or 1\n    lines = ["%s: %d games, %d game-passes | official %.1f%% | lost %.1f points: unfinished levels %.1f (%.0f%% of the "\n             "loss), inefficiency on finished levels %.1f (%.0f%%)"\n             % (label, s["games"], s["passes"], 100 * s["official"], 100 * lost, 100 * s["unfinished"],\n                100 * s["unfinished"] / lost if lost else 0, 100 * s["inefficiency"],\n                100 * s["inefficiency"] / lost if lost else 0),\n             "  first uncleared level per game-pass: " + ", ".join(\n                 "%s %d (%.0f%%)" % (k, st[k], 100.0 * st[k] / tot) for k in ("L1", "L2", "L3+", "all cleared"))]\n    return "\\n".join(lines)\n\n\ndef run(paths, env_dir):\n    baselines = D.load_baselines(env_dir)\n    rows = []\n    for p in paths:\n        for f in sorted(glob.glob(os.path.join(p, "*_events.jsonl"))):\n            r = score_pass(f, baselines)\n            if r:\n                rows.append(r)\n    return rows\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    ap.add_argument("paths", nargs="+")\n    ap.add_argument("--env-dir", default=D.ENV_DIR)\n    ap.add_argument("--per-game", action="store_true")\n    a = ap.parse_args(argv)\n    rows = run(a.paths, a.env_dir)\n    if not rows:\n        print("LOSS_DECOMPOSE no scorable passes (check --env-dir)")\n        return 3\n    s = summarize(rows)\n    print(render(s, "LOSS_DECOMPOSE " + ", ".join(os.path.basename(p.rstrip("/")) for p in a.paths)))\n    if a.per_game:\n        for g, (o, c, n) in sorted(s["per_game"].items(), key=lambda kv: kv[1][0]):\n            print("  %-5s passes=%d official %5.1f%%  unfinished %5.1f  inefficiency %4.1f" % (\n                g, n, 100 * o, 100 * (1 - c), 100 * (c - o)))\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'
open('loss_decompose.py', 'w', encoding='utf-8').write(_SRC)
print('wrote loss_decompose.py', len(_SRC), 'chars')


In [ ]:
# embedded verbatim from fi939-agent-rebuild/paper_public_eval.py
_SRC = '#!/usr/bin/env python3\n"""FI-939 paper: public-evaluation table for every offline screen, in the official metric.\n\nWhy (2026-10-27): the paper rubric asks for public eval-set results beside the leaderboard numbers (open item in the\npaper draft). Every screen already ran the 25 public games x 4 passes; this turns those traces into one table in the\ncompetition\'s own metric (duck_score_estimate / loss_decompose scoring: per completed level min((human/AI)^2, 1.15),\nlevel-number weighted per game, mean over games), with the pass-to-pass spread so a reader can judge the noise.\n\nPer run: official score as the mean of the 4 per-pass scores (each pass = mean over games) and their sd; the same on\nthe 14 held-out games the screen gate uses; total levels per pass; and the share of the lost score that is unfinished\nlevels. Works on full traces or on bundle-only results (it reads only *_events.jsonl).\n\n  paper_public_eval.py LABEL=DIR [LABEL=DIR ...] [--env-dir DIR] [--markdown]\nExit: 0 ok | 2 usage | 3 nothing scorable\n"""\nimport argparse, glob, os, statistics, sys\nfrom collections import defaultdict\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport duck_score_estimate as D  # noqa: E402\nimport loss_decompose as L  # noqa: E402\n\nHELDOUT = ("ar25", "bp35", "cd82", "dc22", "g50t", "ka59", "lp85", "r11l", "re86", "sb26", "sk48", "su15", "tu93", "vc33")\n\n\ndef run_rows(d, baselines):\n    rows = []\n    for f in sorted(glob.glob(os.path.join(d, "**", "*_events.jsonl"), recursive=True)):\n        r = L.score_pass(f, baselines)\n        if r:\n            rows.append(r)\n    return rows\n\n\ndef table_row(rows, heldout=HELDOUT):\n    """Pure. rows from loss_decompose.score_pass -> summary dict."""\n    by_pass = defaultdict(list)\n    for r in rows:\n        by_pass[r["pass"]].append(r)\n\n    def pass_scores(sel):\n        out = []\n        for p, rs in sorted(by_pass.items()):\n            rs = [r for r in rs if sel(r)]\n            if rs:\n                out.append(sum(r["official"] for r in rs) / len(rs))\n        return out\n\n    allp = pass_scores(lambda r: True)\n    hop = pass_scores(lambda r: r["game"] in heldout)\n    lv = [sum(r["levels"] for r in rs) for _, rs in sorted(by_pass.items())]\n    s = L.summarize(rows)\n    lost = 1 - s["official"]\n    sd = lambda xs: statistics.stdev(xs) if len(xs) > 1 else 0.0\n    return {"games": s["games"], "passes": len(by_pass), "official": statistics.mean(allp) if allp else 0.0,\n            "official_sd": sd(allp), "heldout": statistics.mean(hop) if hop else 0.0, "heldout_sd": sd(hop),\n            "levels_per_pass": statistics.mean(lv) if lv else 0.0, "unfinished_share": s["unfinished"] / lost if lost else 0.0}\n\n\ndef render(results, markdown=False):\n    head = ("run", "games", "passes", "official % (sd)", "held-out 14 % (sd)", "levels/pass", "loss from unfinished")\n    lines = []\n    if markdown:\n        lines += ["| " + " | ".join(head) + " |", "|" + "---|" * len(head)]\n    for label, t in results:\n        cells = (label, str(t["games"]), str(t["passes"]), "%.1f (%.1f)" % (100 * t["official"], 100 * t["official_sd"]),\n                 "%.1f (%.1f)" % (100 * t["heldout"], 100 * t["heldout_sd"]), "%.1f" % t["levels_per_pass"],\n                 "%.0f%%" % (100 * t["unfinished_share"]))\n        lines.append(("| " + " | ".join(cells) + " |") if markdown else "  ".join("%-22s" % c if i == 0 else "%-18s" % c\n                                                                                 for i, c in enumerate(cells)))\n    return "\\n".join(lines)\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    ap.add_argument("runs", nargs="+", help="LABEL=DIR")\n    ap.add_argument("--env-dir", default=D.ENV_DIR)\n    ap.add_argument("--markdown", action="store_true")\n    a = ap.parse_args(argv)\n    baselines = D.load_baselines(a.env_dir)\n    results = []\n    for spec in a.runs:\n        if "=" not in spec:\n            print("PUBLIC_EVAL usage: LABEL=DIR, got %r" % spec); return 2\n        label, d = spec.split("=", 1)\n        rows = run_rows(os.path.expanduser(d), baselines)\n        if rows:\n            results.append((label, table_row(rows)))\n        else:\n            print("PUBLIC_EVAL skip %s (no scorable passes)" % label)\n    if not results:\n        print("PUBLIC_EVAL nothing scorable (check --env-dir)"); return 3\n    print(render(results, a.markdown))\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'
open('paper_public_eval.py', 'w', encoding='utf-8').write(_SRC)
print('wrote paper_public_eval.py', len(_SRC), 'chars')


In [ ]:
# embedded verbatim from fi939-agent-rebuild/bench_summary.py
_SRC = '#!/usr/bin/env python3\n"""FI-939 38b: read the FINAL benchmark summary from a Kaggle kernel log and compare per-game levels with our runs.\n\nWhy: on 2026-10-03 the M2 fork\'s commit run was reported as "mean 12.65" because a first-match grep for\n"mean score:" picked up one of the periodic in-progress summaries the TAAF/Duck benchmark prints while games are\nstill running (`ended:     in progress`). The real result, in the last summary whose `ended:` is a timestamp, was\n43.59. The error went into a doc commit and a Slack readout before it was caught. This module parses summaries\nstructurally and only trusts a COMPLETE one (ended = timestamp). It refuses to report when the run never finished.\n\n  bench_summary.py KERNEL_LOG [--compare RUN_DIR ...] [--json]\nKERNEL_LOG: the JSON stream Kaggle returns for `kernels output` (list of {stream_name,time,data}) or plain text.\nRUN_DIR: one of our recorded runs (*_p0_events.jsonl); per-game levels via duck_scorecard.load_run.\nAlso prints PROGRESSION: levels per snapshot and, for the last interval, how many games were still gaining\nlevels (BUDGET-BOUND when at least half were).\nExit: 0 final summary found | 3 only in-progress summaries | 4 no summary | 2 usage\n"""\nimport json, os, re, sys\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, HERE)\n\nHDR = re.compile(r"^benchmark:\\s+\\S+\\s*$", re.M)\nFIELD = re.compile(r"^(games|passes|runs|started|ended|mean score|median score|total actions|total tokens|total wallclock):\\s*(.*?)\\s*$", re.M)\nGAME = re.compile(r"^\\s+([a-z0-9]{2,8}-[0-9a-f]{6,})(?::\\s*score=([\\d.]+),\\s*levels=([\\d.]+)/(\\d+),\\s*actions=(\\d+))?", re.M)\nTS = re.compile(r"^\\d{4}-\\d{2}-\\d{2} \\d{2}:\\d{2}:\\d{2}$")\n\n\ndef log_text(raw):\n    """Kaggle kernel log JSON stream -> concatenated text; plain text passes through."""\n    s = raw.lstrip()\n    if s.startswith("["):\n        try:\n            return "".join(e.get("data", "") for e in json.loads(s))\n        except (ValueError, AttributeError):\n            pass\n    return raw\n\n\ndef summaries(text):\n    """All summary blocks in order: dicts with fields + per-game list + complete flag."""\n    starts = [m.start() for m in HDR.finditer(text)]\n    out = []\n    for k, a in enumerate(starts):\n        b = starts[k + 1] if k + 1 < len(starts) else len(text)\n        block = text[a:b]\n        f = {m.group(1): m.group(2) for m in FIELD.finditer(block)}\n        if "mean score" not in f:\n            continue\n        pg_at = block.find("per-game")\n        games = []\n        if pg_at >= 0:\n            for m in GAME.finditer(block[pg_at:]):\n                if m.group(2) is None:\n                    break   # truncated line (mid-run print cut off)\n                games.append({"game": m.group(1), "score": float(m.group(2)), "levels": float(m.group(3)),\n                              "max_levels": int(m.group(4)), "actions": int(m.group(5))})\n        out.append({"fields": f, "games": games, "complete": bool(TS.match(f.get("ended", "")))})\n    return out\n\n\ndef final_summary(text):\n    """The LAST complete summary, or None."""\n    done = [s for s in summaries(text) if s["complete"]]\n    return done[-1] if done else None\n\n\ndef compare(final, run_levels):\n    """final: summary dict; run_levels: {name: {game_prefix: levels}} -> rows over the summary\'s games."""\n    rows = []\n    for g in final["games"]:\n        pre = g["game"].split("-")[0]\n        row = {"game": pre, "bench": g["levels"]}\n        for name, lv in run_levels.items():\n            row[name] = lv.get(pre)\n        rows.append(row)\n    return rows\n\n\ndef levels_of_run(run_dir):\n    import duck_scorecard as D\n    return {g.split("-")[0]: r["levels"] for g, r in D.load_run(os.path.expanduser(run_dir)).items()}\n\n\ndef progression(allsum):\n    """Is the run budget-bound? Uses every summary (in-progress ones included) in log order.\n    Per game: levels and actions at each snapshot. In the LAST interval (second-to-last summary -> last), a game is\n    \'climbing\' if it gained levels, \'active\' if it took actions but gained none, \'stopped\' if it took no actions\n    (its own budget or the game ended earlier). Many climbing games at the end means more time or tokens would still\n    buy levels: the score is limited by the budget, not by the agent running out of ideas."""\n    snaps = [x for x in allsum if x["games"]]\n    if len(snaps) < 2:\n        return None\n    a, b = snaps[-2], snaps[-1]\n    prev = {g["game"]: g for g in a["games"]}\n    per, state = [], {"climbing": 0, "active": 0, "stopped": 0}\n    for g in b["games"]:\n        p = prev.get(g["game"])\n        if p is None:\n            continue\n        st = "climbing" if g["levels"] > p["levels"] else ("active" if g["actions"] > p["actions"] else "stopped")\n        state[st] += 1\n        per.append({"game": g["game"].split("-")[0], "levels": [x_lv(x, g["game"]) for x in snaps], "state": st})\n    tot = [sum(g["levels"] for g in x["games"]) for x in snaps]\n    wall = [x["fields"].get("total wallclock", "") for x in snaps]\n    return {"snapshots": len(snaps), "total_levels": tot, "wallclock": wall, "last_interval": state,\n            "last_interval_levels": tot[-1] - tot[-2], "per_game": per}\n\n\ndef x_lv(summary, game):\n    for g in summary["games"]:\n        if g["game"] == game:\n            return g["levels"]\n    return None\n\n\ndef main(argv):\n    as_json = "--json" in argv\n    argv = [a for a in argv if a != "--json"]\n    if not argv:\n        print(__doc__); return 2\n    runs = []\n    if "--compare" in argv:\n        i = argv.index("--compare")\n        runs, argv = argv[i + 1:], argv[:i]\n    if len(argv) != 1:\n        print(__doc__); return 2\n    text = log_text(open(os.path.expanduser(argv[0]), encoding="utf-8", errors="replace").read())\n    allsum = summaries(text)\n    if not allsum:\n        print("NO_SUMMARY"); return 4\n    fin = final_summary(text)\n    if fin is None:\n        print("IN_PROGRESS_ONLY: %d summaries, none with a timestamp \'ended:\' (last mean %s)" % (\n            len(allsum), allsum[-1]["fields"].get("mean score")))\n        return 3\n    f = fin["fields"]\n    lv = sum(g["levels"] for g in fin["games"]); mx = sum(g["max_levels"] for g in fin["games"])\n    res = {"summaries": len(allsum), "ended": f.get("ended"), "games": len(fin["games"]), "mean": f.get("mean score"),\n           "median": f.get("median score"), "actions": f.get("total actions"), "levels": lv, "max_levels": mx}\n    rows = compare(fin, {os.path.basename(os.path.normpath(r)): levels_of_run(r) for r in runs}) if runs else []\n    prog = progression(allsum)\n    if as_json:\n        print(json.dumps({"final": res, "per_game": fin["games"], "compare": rows, "progression": prog}, indent=1))\n        return 0\n    print("FINAL ended=%s games=%d mean=%s median=%s actions=%s levels=%g/%d (summaries in log: %d)" % (\n        res["ended"], res["games"], res["mean"], res["median"], res["actions"], lv, mx, len(allsum)))\n    if prog:\n        li = prog["last_interval"]\n        print("PROGRESSION levels by snapshot %s (wallclock %s); last interval +%g levels: climbing=%d active=%d "\n              "stopped=%d%s" % (prog["total_levels"], prog["wallclock"], prog["last_interval_levels"], li["climbing"],\n                                li["active"], li["stopped"],\n                                " -> BUDGET-BOUND" if li["climbing"] * 2 >= sum(li.values()) else ""))\n    if rows:\n        names = [os.path.basename(os.path.normpath(r)) for r in runs]\n        print("game    bench  " + "  ".join("%14s" % n[:14] for n in names))\n        tot = {n: 0 for n in names}; btot = 0\n        for r in rows:\n            btot += r["bench"]\n            cells = []\n            for n in names:\n                v = r[n]\n                cells.append("%14s" % ("-" if v is None else "%g" % v))\n                tot[n] += v or 0\n            print("%-6s %6g  %s" % (r["game"], r["bench"], "  ".join(cells)))\n        print("TOTAL  %6g  %s" % (btot, "  ".join("%14g" % tot[n] for n in names)))\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main(sys.argv[1:]))\n'
open('bench_summary.py', 'w', encoding='utf-8').write(_SRC)
print('wrote bench_summary.py', len(_SRC), 'chars')


In [ ]:
# embedded verbatim from fi939-agent-rebuild/experiment_ledger.py
_SRC = '#!/usr/bin/env python3\n"""FI-939 experiment ledger: group every scored Kaggle submission by the exact kernel\nscript version it came from, and report n / mean / range per version.\n\nWHY: each version was scored once, so 0.34 vs 0.15 vs 0.04 could partly be rerun noise.\nThe daily tick now resubmits unchanged kernels (REPEAT-SAMPLE) to measure that noise. This\ntool reads the scored history (READ-ONLY; never submits) and says whether a gap between two\nversions exceeds the largest spread any single unchanged version has shown so far.\n\nGrouping key: (kernel slug, scriptVersionId) parsed from each submission\'s URL, which\nidentifies the exact notebook version that was scored. Descriptions are free text and are\nused only for a display label.\n\nUsage:\n  experiment_ledger.py                       # live, via the kaggle API (read-only)\n  experiment_ledger.py --from-file dump.json # offline, from a saved submissions dump\n  options: --labels labels.json ({scriptVersionId: label}), --json, --compare SID_A SID_B,\n           --base KERNEL_SLUG (every later kernel vs that base, with a level-aware threshold)\nExit: 0 ok, 2 input unreadable, 3 live fetch failed.\n"""\nimport argparse\nimport json\nimport os\nimport re\nimport sys\n\nCOMPETITION = "arc-prize-2026-arc-agi-3"\n_URL_RE = re.compile(r"/code/[^/]+/([^?]+)\\?scriptVersionId=(\\d+)")\n\n\ndef extract_records(text):\n    """Return the submissions list from text that may carry CLI warning lines first."""\n    i = text.find("[")\n    while i != -1:\n        try:\n            val = json.loads(text[i:])\n            if isinstance(val, list):\n                return val\n        except ValueError:\n            pass\n        i = text.find("[", i + 1)\n    return []\n\n\ndef _label_from_desc(desc):\n    d = (desc or "").strip()\n    head = re.split(r"[:;(]", d, 1)[0].strip()\n    return head[:40] if head else "?"\n\n\ndef normalize(r, labels=None):\n    url = str(r.get("url") or "")\n    m = _URL_RE.search(url)\n    kernel, sid = (m.group(1), m.group(2)) if m else ("", "")\n    status = str(r.get("status") or "")\n    raw = str(r.get("public_score") or "").strip()\n    try:\n        score = float(raw) if raw and raw.lower() != "none" else None\n    except ValueError:\n        score = None\n    scored = score is not None and status.upper().endswith("COMPLETE")\n    # FI-939 38eg: a COMPLETE draw at exactly 0.00 is a failed scored rerun (the 10-22 default and 10-24 serverguard\n    # zeros, byte-identical 3,411-byte files), not a sample of the policy. Counted as a score it made the default\'s\n    # spread 29.10 and its sd ~8, so every arm read "within_noise" and the power plan asked for far too many repeats.\n    # It is reported as failed and kept out of every mean, spread and sd.\n    failed = scored and score == 0.0\n    scored = scored and not failed\n    label = (labels or {}).get(sid) or _label_from_desc(r.get("description"))\n    return {"date": str(r.get("date") or ""), "kernel": kernel, "script_version": sid,\n            "score": score if scored else None, "scored": scored, "failed": failed, "status": status,\n            "ref": str(r.get("ref") or ""), "bytes": str(r.get("total_bytes") or ""),\n            "label": label, "description": str(r.get("description") or "")}\n\n\ndef group(entries):\n    by = {}\n    for e in entries:\n        key = (e["kernel"], e["script_version"])\n        by.setdefault(key, []).append(e)\n    out = []\n    for (kernel, sid), es in by.items():\n        es.sort(key=lambda x: x["date"])\n        scores = [x["score"] for x in es if x["scored"]]\n        n = len(scores)\n        failed = sum(1 for x in es if x.get("failed"))\n        out.append({\n            "kernel": kernel, "script_version": sid, "label": es[0]["label"],\n            "n": n, "failed": failed, "pending": len(es) - n - failed,\n            "scores": scores,\n            "mean": (sum(scores) / n) if n else None,\n            "min": min(scores) if n else None,\n            "max": max(scores) if n else None,\n            "range": (max(scores) - min(scores)) if n >= 2 else None,\n            "first_date": es[0]["date"], "last_date": es[-1]["date"],\n        })\n    out.sort(key=lambda g: g["first_date"])\n    return out\n\n\ndef noise_floor(groups):\n    ranges = [g["range"] for g in groups if g.get("range") is not None]\n    # FI-939 2026-10-04: the only repeats so far sit at 3-4 points; the M2 base scores ~25. An absolute spread\n    # measured at one level says little about another, so the relative spread (range / mean) is kept too.\n    rel = [g["range"] / g["mean"] for g in groups if g.get("range") is not None and g.get("mean")]\n    return {"max_range": max(ranges) if ranges else None, "groups_with_repeats": len(ranges),\n            "max_rel_range": max(rel) if rel else None}\n\n\ndef compare(a, b, nf):\n    if a.get("mean") is None or b.get("mean") is None:\n        return {"delta": None, "verdict": "unscored"}\n    delta = a["mean"] - b["mean"]\n    if nf.get("max_range") is None:\n        verdict = "no_variance_data"\n    elif abs(delta) > nf["max_range"] + 1e-12:\n        verdict = "exceeds_observed_spread"\n    else:\n        verdict = "within_observed_spread"\n    return {"delta": delta, "verdict": verdict, "a": a.get("script_version"), "b": b.get("script_version")}\n\n\ndef load_external(path, kernel):\n    """Scores of runs of the SAME configuration as `kernel` made by others (public leaderboard), from a JSON file\n    {"<kernel slug>": {"scores": [..], "evidence": "..."}}. Only used for the base\'s spread (noise threshold).\n    Missing file or kernel -> []."""\n    try:\n        doc = json.load(open(path))\n        return [float(x) for x in ((doc.get(kernel) or {}).get("scores") or [])]\n    except (OSError, ValueError, TypeError, AttributeError):\n        return []\n\n\ndef load_references(path, kernel):\n    """FI-939 38bn: hidden scores of the SAME code run from OTHER accounts, kept apart from noise samples. They are\n    never pooled into the base\'s spread (38s retired them as noise because our own repeats were tight); they are shown\n    next to the base so the identical-code gap stays visible. File: {"references": {"<kernel>": [{"score", "who",\n    "seen"}]}}. Missing -> []."""\n    try:\n        doc = json.load(open(path))\n        out = []\n        for r in ((doc.get("references") or {}).get(kernel) or []):\n            out.append({"score": float(r["score"]), "who": str(r.get("who", "?")), "seen": str(r.get("seen", "?"))})\n        return out\n    except (OSError, ValueError, TypeError, AttributeError, KeyError):\n        return []\n\n\ndef reference_gap(base_scores, refs):\n    """(ref mean - base mean, ref min, ref max) or None."""\n    if not base_scores or not refs:\n        return None\n    xs = [r["score"] for r in refs]\n    return (sum(xs) / len(xs) - sum(base_scores) / len(base_scores), min(xs), max(xs))\n\n\nZ_ALPHA, Z_BETA = 1.96, 0.8416  # two-sided 5% test, 80% power\n\n\ndef stdev(xs):\n    xs = [float(x) for x in xs]\n    if len(xs) < 2:\n        return None\n    m = sum(xs) / len(xs)\n    return (sum((x - m) ** 2 for x in xs) / (len(xs) - 1)) ** 0.5\n\n\ndef samples_needed(sd, delta, z_a=Z_ALPHA, z_b=Z_BETA):\n    """Hidden scores needed PER SIDE to detect a true mean difference `delta` between an arm and the base, for\n    run-to-run standard deviation `sd` (normal approximation, two-sample, equal n). FI-939 2026-10-04: identical M2\n    configs scored 25.54 .. 31.47, so the planning question is how many repeats an arm needs, not whether n=2\n    suffices. Returns None if sd or delta is unusable."""\n    if not sd or not delta or delta <= 0:\n        return None\n    import math\n    return max(2, math.ceil(2 * ((z_a + z_b) * sd / delta) ** 2))\n\n\n# Lower 2.5% quantiles of chi-square for df = 1..10 (tables); beyond 10, Wilson-Hilferty.\n_CHI2_LO_025 = {1: 0.000982, 2: 0.0506, 3: 0.2158, 4: 0.4844, 5: 0.8312, 6: 1.2373, 7: 1.6899, 8: 2.1797,\n                9: 2.7004, 10: 3.2470}\n\n\ndef sd_upper(sd, n):\n    """FI-939 38bm: one-sided-in-effect 95% upper bound for a normal sd estimated from n samples (df = n - 1), i.e.\n    sd * sqrt(df / chi2_{0.025, df}). With n = 2 the bound is about 32x the estimate: two identical scores do NOT\n    show the noise is small, only that it might be. Returns None for unusable input."""\n    if sd is None or n is None or n < 2:\n        return None\n    df = n - 1\n    q = _CHI2_LO_025.get(df)\n    if q is None:\n        z = 1.959964\n        q = df * (1 - 2.0 / (9 * df) - z * (2.0 / (9 * df)) ** 0.5) ** 3\n    return sd * (df / q) ** 0.5\n\n\ndef pooled_sd(groups, near_mean, rel=0.3):\n    """FI-939 38cl: pooled within-version run-to-run sd over every version scored at least twice whose mean lies\n    within +-rel of `near_mean` (same score regime; a 3-point version says little about a 25-point one).\n    Returns (sd, df, [(kernel, scores)]) or None. Why: the base\'s two scores agreed to 0.03 and the power plan rested\n    on them, then one public kernel scored 28.49 and 23.63 on consecutive days; the base alone hid the noise."""\n    if not near_mean:\n        return None\n    ss, df, used = 0.0, 0, []\n    for g in groups:\n        xs = g.get("scores") or []\n        if len(xs) < 2 or g.get("mean") is None or abs(g["mean"] - near_mean) > rel * abs(near_mean):\n            continue\n        m = sum(xs) / len(xs)\n        ss += sum((x - m) ** 2 for x in xs)\n        df += len(xs) - 1\n        used.append((g["kernel"], xs))\n    if df == 0:\n        return None\n    return (ss / df) ** 0.5, df, used\n\n\ndef vs_base(groups, base_kernel, nf, min_n=2, external=None):\n    """Each kernel scored since the base\'s first submission vs the base (all versions of each kernel pooled).\n    Threshold: once the base has a repeat, the larger of the absolute floor and the base\'s own spread; before that,\n    the larger of the absolute floor and the relative floor x the base mean (conservative: the relative floor comes\n    from lower-score repeats). Verdict needs_samples until both sides have >= min_n scores."""\n    pooled = {}\n    for g in groups:\n        p = pooled.setdefault(g["kernel"], {"kernel": g["kernel"], "label": g["label"], "scores": [],\n                                             "first_date": g["first_date"]})\n        p["scores"].extend(g["scores"])\n        p["first_date"] = min(p["first_date"], g["first_date"])\n    base = pooled.get(base_kernel)\n    if base is None or not base["scores"]:\n        return None, []\n    bm = sum(base["scores"]) / len(base["scores"])\n    # FI-939 2026-10-04: identical-config public reruns (verified with notebook_env_diff) measure the same noise; they\n    # widen the spread sample but never move the base mean or count toward the base\'s own n.\n    bs = base["scores"] + list(external or [])\n    if len(bs) >= 2:\n        # the base has its own repeat: its measured spread at its own level (never below the absolute floor)\n        thr = [x for x in (nf.get("max_range"), max(bs) - min(bs)) if x is not None]\n    else:\n        # no repeat at this level yet: scale the relative floor (conservative, from lower-score repeats)\n        thr = [x for x in (nf.get("max_range"), (nf.get("max_rel_range") or 0) * bm if nf.get("max_rel_range") else None)\n               if x is not None]\n    threshold = max(thr) if thr else None\n    rows = []\n    for k, p in sorted(pooled.items(), key=lambda kv: kv[1]["first_date"]):\n        if k == base_kernel or p["first_date"] < base["first_date"] or not p["scores"]:\n            continue\n        m = sum(p["scores"]) / len(p["scores"])\n        d = m - bm\n        if len(p["scores"]) < min_n or len(base["scores"]) < min_n:\n            v = "needs_samples"\n        elif threshold is None:\n            v = "no_variance_data"\n        elif abs(d) > threshold:\n            v = "better" if d > 0 else "worse"\n        else:\n            v = "within_noise"\n        rows.append({"kernel": k, "label": p["label"], "n": len(p["scores"]), "mean": m, "delta": d, "verdict": v})\n    head = {"kernel": base_kernel, "n": len(base["scores"]), "mean": bm, "threshold": threshold,\n            "external_n": len(external or [])}\n    return head, rows\n\n\ndef _f(x):\n    return "-" if x is None else "%.2f" % x\n\n\ndef render(groups, nf):\n    lines = ["%-10s %-22s %-24s %-5s %-6s %-11s %s" % ("first", "kernel", "label", "n", "mean", "min-max", "scriptVersionId")]\n    for g in groups:\n        rng = "%s-%s" % (_f(g["min"]), _f(g["max"])) if g["n"] else ("pending" if g["pending"] else "-")\n        lines.append("%-10s %-22s %-24s %-5s %-6s %-11s %s" % (\n            g["first_date"][:10], g["kernel"][:22], g["label"][:24], "n=%d" % g["n"], _f(g["mean"]), rng,\n            g["script_version"] + (" (+%d failed 0.00)" % g["failed"] if g.get("failed") else "")))\n    nfail = sum(g.get("failed", 0) for g in groups)\n    if nfail:\n        lines.append("Failed scored runs (0.00, excluded from every mean/spread/sd): %d" % nfail)\n    if nf["max_range"] is None:\n        lines.append("Noise floor: no version has been scored twice yet; every gap is a single-sample comparison.")\n    else:\n        lines.append("Noise floor: largest spread for an unchanged version = %.2f across %d repeated version(s)."\n                     % (nf["max_range"], nf["groups_with_repeats"]))\n    lines.append("Caveat: with so few repeats, gaps are suggestive, not conclusive; a floor from n=2 understates true variance.")\n    return "\\n".join(lines)\n\n\ndef collect_pages(fetch_page, page_size=100, max_pages=20):\n    """Read every page of a paged list call. KaggleApi.competition_submissions returns 20 rows per call by default\n    and gives no next-page token, so one call silently dropped older draws (8 of the default\'s 10 on 2026-10-24).\n    Stops on a short page or a page with nothing new; refuses (RuntimeError) to truncate past max_pages."""\n    seen, out = set(), []\n    for page in range(1, max_pages + 1):\n        batch = [x for x in (fetch_page(page, page_size) or []) if x is not None]\n        fresh = [x for x in batch if str(getattr(x, "ref", "")) not in seen]\n        for x in fresh:\n            seen.add(str(getattr(x, "ref", "")))\n        out.extend(fresh)\n        if len(batch) < page_size or not fresh:\n            return out\n    raise RuntimeError("more than %d pages of submissions; raise max_pages" % max_pages)\n\n\ndef fetch_live():\n    from kaggle.api.kaggle_api_extended import KaggleApi  # read-only list call\n    api = KaggleApi()\n    api.authenticate()\n    out = []\n    for s in collect_pages(lambda n, k: api.competition_submissions(COMPETITION, page_number=n, page_size=k)):\n        out.append({k: str(getattr(s, k, "") or "") for k in ("date", "description", "public_score", "status", "url",\n                                                              "ref", "total_bytes")})\n    return out\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    ap.add_argument("--from-file")\n    ap.add_argument("--labels")\n    ap.add_argument("--json", action="store_true")\n    ap.add_argument("--compare", nargs=2, metavar=("SID_A", "SID_B"))\n    ap.add_argument("--base", metavar="KERNEL_SLUG", help="judge every later kernel against this base kernel")\n    ap.add_argument("--power", nargs="*", type=float, metavar="DELTA",\n                    help="with --base: hidden scores needed per side to detect these mean gains (default 1 2 3 4 6)")\n    ap.add_argument("--external", default=os.path.join(os.path.dirname(os.path.abspath(__file__)),\n                                                         "duck_external_samples.json"),\n                    help="identical-config public scores per kernel (base spread only)")\n    a = ap.parse_args(argv)\n    labels = {}\n    try:\n        if a.labels:\n            with open(a.labels) as f:\n                labels = json.load(f)\n        if a.from_file:\n            with open(a.from_file, encoding="utf-8") as f:\n                recs = extract_records(f.read())\n        else:\n            recs = None\n    except (OSError, ValueError) as ex:\n        print("LEDGER_INPUT_ERROR: %s" % ex, file=sys.stderr)\n        return 2\n    if recs is None:\n        try:\n            recs = fetch_live()\n        except Exception as ex:  # network / auth\n            print("LEDGER_FETCH_ERROR: %s" % ex, file=sys.stderr)\n            return 3\n    groups = group([normalize(r, labels) for r in recs])\n    nf = noise_floor(groups)\n    cmp_out = None\n    if a.compare:\n        idx = {g["script_version"]: g for g in groups}\n        ga, gb = idx.get(a.compare[0]), idx.get(a.compare[1])\n        cmp_out = compare(ga or {}, gb or {}, nf)\n    if a.json:\n        print(json.dumps({"groups": groups, "noise_floor": nf, "compare": cmp_out}, indent=1))\n    else:\n        print(render(groups, nf))\n        if cmp_out:\n            print("Compare %s vs %s: delta=%s -> %s" % (a.compare[0], a.compare[1], _f(cmp_out["delta"]), cmp_out["verdict"]))\n        if a.base:\n            head, rows = vs_base(groups, a.base, nf, external=load_external(a.external, a.base))\n            if head is None:\n                print("Base %s: no scored submission" % a.base)\n            else:\n                print("Base %s: n=%d mean=%s; decision threshold %s (base\'s own spread once repeated,"\n                      " else the relative floor x base mean; never below the absolute floor; +%d identical-config public scores)" % (\n                          head["kernel"], head["n"], _f(head["mean"]), _f(head["threshold"]), head["external_n"]))\n                for r in rows:\n                    print("  %-28s n=%d mean=%s delta=%s -> %s" % (r["kernel"][:28], r["n"], _f(r["mean"]),\n                                                                 _f(r["delta"]), r["verdict"]))\n                refs = load_references(a.external, a.base)\n                if refs:\n                    own = [x for g in groups if g["kernel"] == a.base for x in g["scores"]]\n                    gap = reference_gap(own, refs)\n                    print("Reference: identical code run from other accounts (NOT pooled as noise): %s" % ", ".join(\n                        "%.2f %s (seen %s)" % (r["score"], r["who"], r["seen"]) for r in refs))\n                    if gap:\n                        print("  gap: reference mean - our base mean = %+.2f (references %.2f..%.2f, our base n=%d)" % (\n                            gap[0], gap[1], gap[2], len(own)))\n                if a.power is not None:\n                    pool = [x for g in groups if g["kernel"] == a.base for x in g["scores"]] + \\\n                        load_external(a.external, a.base)\n                    sd = stdev(pool)\n                    if sd is None:\n                        print("Power: need >= 2 base samples (own + identical-config public) for a spread")\n                    else:\n                        print("Power: base run-to-run sd %.2f from %d samples [%s]" % (\n                            sd, len(pool), ", ".join("%.2f" % x for x in sorted(pool))))\n                        up = sd_upper(sd, len(pool))\n                        print("  95%% upper bound on that sd: %.2f (df=%d) - plan with both columns" % (up, len(pool) - 1))\n                        for d in (a.power or [1, 2, 3, 4, 6]):\n                            print("  detect +%.1f -> %d hidden scores per side at sd %.2f; %d at the upper bound %.2f "\n                                  "[5%% two-sided, 80%% power]" % (d, samples_needed(sd, d), sd,\n                                                                   samples_needed(up, d), up))\n                    ps = pooled_sd(groups, head["mean"])\n                    if ps:\n                        psd, pdf, used = ps\n                        pup = sd_upper(psd, pdf + 1)\n                        print("Power (pooled, 38cl): within-version sd %.2f, df=%d, from %s; 95%% upper bound %.2f" % (\n                            psd, pdf, "; ".join("%s [%s]" % (k[:28], ", ".join("%.2f" % x for x in xs)) for k, xs in used),\n                            pup))\n                        for d in (a.power or [1, 2, 3, 4, 6]):\n                            print("  detect +%.1f -> %d hidden scores per side at pooled sd %.2f; %d at the upper bound"\n                                  " %.2f" % (d, samples_needed(psd, d), psd, samples_needed(pup, d), pup))\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'
open('experiment_ledger.py', 'w', encoding='utf-8').write(_SRC)
print('wrote experiment_ledger.py', len(_SRC), 'chars')


In [ ]:
# embedded verbatim from fi939-agent-rebuild/rank_calibration.py
_SRC = '#!/usr/bin/env python3\n"""FI-939: do offline commit-run results rank our kernels the way the hidden leaderboard does?\n\nWhy: offline scores (~46) and hidden scores (~25.5) differ by construction (different games, different load; DUCK\n38w/38x), so the offset itself is not a target. What matters is whether an offline win predicts a hidden win. Earlier\nproxies did not (DUCK 35, 36y), and one loadsim pair only detects ~8.5 points. Each daily submission is the scarce\nresource (about 27 left before 11-02). If offline rank tracks hidden rank, offline runs can screen arms before they\nuse a slot; if not, they stay a crash/regression check only. This gives that answer from data, updated as each\nhidden score lands (proposed to the operator 10-05).\n\nJoins, per kernel slug:\n  hidden: mean of the COMPLETE scores of all its scored submissions (experiment_ledger grouping, by URL)\n  offline: the FINAL summary mean (and levels) of its latest commit-run log in the m2_commit_watch cache\nReports every joined kernel, then pairwise order agreement: for each pair, does offline order = hidden order? Pairs\nwhose hidden gap is below --noise (default 1.48, the provisional threshold) are counted as TIES, not agreements or\ndisagreements. Spearman rho over all joined kernels is printed when n >= 4.\nFinal line: CALIB_RESULT agree=A disagree=D ties=T of P pairs, rho=R | CALIB_INSUFFICIENT <why>.\nUsage: rank_calibration.py [--from-file submissions.json] [--cache ~/.m2_commit_watch] [--metric score|levels]\n                           [--noise 1.48] [--only-prefix duck-franzen-m2] [--log SLUG=PATH ...]\n"""\nimport argparse, glob, json, os, sys\nfrom itertools import combinations\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, HERE)\nimport bench_summary as B  # noqa: E402\nimport experiment_ledger as L  # noqa: E402\n\n\ndef hidden_by_kernel(records):\n    """{slug: (mean, n)} over every scored submission of the slug (all script versions pooled)."""\n    out = {}\n    for g in L.group([L.normalize(r) for r in records]):\n        if not g["kernel"] or not g["n"]:\n            continue\n        s, n = out.get(g["kernel"], (0.0, 0))\n        out[g["kernel"]] = (s + sum(g["scores"]), n + g["n"])\n    return {k: (s / n, n) for k, (s, n) in out.items()}\n\n\ndef offline_value(path, metric="score"):\n    """Mean per-game score (or levels) of the FINAL summary in a kernel log, or None."""\n    s = B.final_summary(B.log_text(open(path, encoding="utf-8", errors="replace").read()))\n    if not s or not s.get("complete"):\n        return None\n    games = s.get("games") or []\n    key = "levels" if metric == "levels" else "score"\n    vals = [float(g[key]) for g in games if g.get(key) is not None]\n    return sum(vals) / len(vals) if vals else None\n\n\ndef offline_by_kernel(cache, metric="score", extra=None):\n    """{slug: value} from FINAL summaries of <cache>/<slug>/<slug>.log, plus extra {slug: path} logs (older eras\n    kept their logs elsewhere, e.g. ~/klog_fn1/duck-qwen38-flashnext.log); levels = mean levels per game."""\n    out = {}\n    for slug, path in (extra or {}).items():\n        if os.path.exists(os.path.expanduser(path)):\n            v = offline_value(os.path.expanduser(path), metric)\n            if v is not None:\n                out[slug] = v\n    for p in glob.glob(os.path.join(os.path.expanduser(cache), "*", "*.log")):\n        slug = os.path.basename(os.path.dirname(p))\n        if os.path.basename(p) != slug + ".log":\n            continue\n        s = B.final_summary(B.log_text(open(p, encoding="utf-8", errors="replace").read()))\n        if not s or not s.get("complete"):\n            continue\n        games = s.get("games") or []\n        if metric == "levels":\n            vals = [float(g["levels"]) for g in games if g.get("levels") is not None]\n        else:\n            vals = [float(g["score"]) for g in games if g.get("score") is not None]\n        if vals:\n            out[slug] = sum(vals) / len(vals)\n    return out\n\n\ndef ranks(values):\n    order = sorted(range(len(values)), key=lambda i: values[i])\n    r = [0.0] * len(values)\n    i = 0\n    while i < len(order):\n        j = i\n        while j + 1 < len(order) and values[order[j + 1]] == values[order[i]]:\n            j += 1\n        for k in range(i, j + 1):\n            r[order[k]] = (i + j) / 2.0 + 1\n        i = j + 1\n    return r\n\n\ndef spearman(xs, ys):\n    if len(xs) < 2:\n        return None\n    rx, ry = ranks(xs), ranks(ys)\n    mx, my = sum(rx) / len(rx), sum(ry) / len(ry)\n    num = sum((a - mx) * (b - my) for a, b in zip(rx, ry))\n    den = (sum((a - mx) ** 2 for a in rx) * sum((b - my) ** 2 for b in ry)) ** 0.5\n    return num / den if den else None\n\n\ndef pairs(rows, noise):\n    agree = disagree = ties = 0\n    detail = []\n    for a, b in combinations(rows, 2):\n        dh = a["hidden"] - b["hidden"]\n        do = a["offline"] - b["offline"]\n        if abs(dh) < noise:\n            ties += 1\n            kind = "TIE"\n        elif dh * do > 0:\n            agree += 1\n            kind = "AGREE"\n        else:\n            disagree += 1\n            kind = "DISAGREE"\n        detail.append((a["kernel"], b["kernel"], dh, do, kind))\n    return agree, disagree, ties, detail\n\n\ndef run(records, cache, metric="score", noise=1.48, only_prefix=None, extra=None):\n    hid = hidden_by_kernel(records)\n    off = offline_by_kernel(cache, metric, extra)\n    rows = [{"kernel": k, "hidden": hid[k][0], "n": hid[k][1], "offline": off[k]}\n            for k in sorted(set(hid) & set(off)) if not only_prefix or k.startswith(only_prefix)]\n    lines = ["  %-36s hidden %6.2f (n=%d)  offline %s %6.2f" % (r["kernel"], r["hidden"], r["n"], metric, r["offline"])\n             for r in sorted(rows, key=lambda r: -r["hidden"])]\n    missing = sorted(k for k in hid if k not in off and (not only_prefix or k.startswith(only_prefix)))\n    if missing:\n        lines.append("  hidden but no offline FINAL in cache: " + ", ".join(missing))\n    if len(rows) < 2:\n        lines.append("CALIB_INSUFFICIENT %d kernel(s) have both a hidden score and an offline FINAL" % len(rows))\n        return lines\n    a, d, t, detail = pairs(rows, noise)\n    for x, y, dh, do, kind in detail:\n        lines.append("  %-8s %s vs %s: hidden %+.2f, offline %+.2f" % (kind, x, y, dh, do))\n    rho = spearman([r["offline"] for r in rows], [r["hidden"] for r in rows]) if len(rows) >= 4 else None\n    lines.append("CALIB_RESULT agree=%d disagree=%d ties=%d of %d pairs (noise %.2f), rho=%s, n=%d kernels"\n                 % (a, d, t, a + d + t, noise, ("%.2f" % rho) if rho is not None else "n/a (<4)", len(rows)))\n    return lines\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    ap.add_argument("--from-file")\n    ap.add_argument("--cache", default="~/.m2_commit_watch")\n    ap.add_argument("--metric", choices=("score", "levels"), default="score")\n    ap.add_argument("--noise", type=float, default=1.48)\n    ap.add_argument("--only-prefix")\n    ap.add_argument("--log", action="append", default=[], help="SLUG=PATH, an extra offline log (repeatable)")\n    a = ap.parse_args(argv)\n    try:\n        records = (L.extract_records(open(a.from_file, encoding="utf-8").read()) if a.from_file else L.fetch_live())\n    except Exception as ex:  # live fetch or file read; report, do not crash the scheduled caller\n        print("CALIB_ERROR %s" % str(ex)[:200])\n        return 3\n    extra = dict(x.split("=", 1) for x in a.log if "=" in x)\n    print("\\n".join(run(records, a.cache, a.metric, a.noise, a.only_prefix, extra)))\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'
open('rank_calibration.py', 'w', encoding='utf-8').write(_SRC)
print('wrote rank_calibration.py', len(_SRC), 'chars')


In [ ]:
# embedded verbatim from fi939-agent-rebuild/screen_calibration.py
_SRC = '#!/usr/bin/env python3\n"""FI-939: does the offline HF screen predict the hidden leaderboard?\n\nWhy (2026-10-28): every arm is now screened offline (25 public games x 4 passes, ~US$24) before it may use one of the\nfew daily hidden submissions, and the screen decides which arms reach a slot (DUCK 38ex). That rule is only as good as\nthe screen\'s link to the hidden score, and nothing measured it: rank_calibration.py answers the same question for the\nolder commit-run logs, not for the HF screens. The paper needs the answer too ("does the screen predict hidden?"), and\nthe 10-28 deep140 draw was kept as a pre-registered test of exactly this (DUCK 38ez).\n\nJoins, per kernel slug (screen_kernel_map.json: slug -> the HF trace dirs screened for that exact config):\n  hidden: mean of all scored submissions of the slug (experiment_ledger grouping), pooled over the slugs listed in the\n          map\'s hidden_pool for that slug; 0.00 draws (failed runs) are dropped unless --keep-zero\n  screen: official-metric public score (paper_public_eval), the mean over the mapped replicate dirs\nReports each kernel against the base kernel (hidden delta vs screen delta and whether the signs agree, with hidden\ndeltas inside --noise counted as ties), pairwise order agreement over all joined kernels, Spearman rho when n >= 4,\nand kernels screened but without a usable hidden score (pending, or every draw failed), each with its screen delta:\nthe prediction on record.\nFinal line (cmd-safe): SCREEN_CALIB vs-base agree=A disagree=D ties=T; pairs agree=.. disagree=.. ties=.., noise N; rho=.. n=..\n         or SCREEN_CALIB_INSUFFICIENT <why>.\nUsage: screen_calibration.py [--from-file submissions.json] [--map screen_kernel_map.json] [--traces DIR]\n                             [--env-dir DIR] [--noise 1.48]\nRead-only: lists submissions (or reads a saved list) and reads traces; it never submits.\n"""\nimport argparse, json, os, sys\n\nHERE = os.path.dirname(os.path.abspath(__file__))\nsys.path.insert(0, HERE)\nimport rank_calibration as R  # noqa: E402\n\nMAP_FILE = os.path.join(HERE, "screen_kernel_map.json")\nDEFAULT_TRACES = os.path.expanduser(r"~\\m2\\traces") if os.name == "nt" else os.path.expanduser("~/m2/traces")\n\n\ndef load_map(path=MAP_FILE, with_pools=False):\n    with open(path, encoding="utf-8") as f:\n        d = json.load(f)\n    kernels = {k: list(v) for k, v in d["kernels"].items()}\n    base = d["base"]\n    if base not in kernels:\n        raise ValueError("base kernel %s is not in the map" % base)\n    pools = {k: list(v) for k, v in (d.get("hidden_pool") or {}).items()}\n    return (base, kernels, pools) if with_pools else (base, kernels)\n\n\ndef hidden_scores(records, L=None):\n    """{slug: [complete scores]} over every scored submission (experiment_ledger grouping)."""\n    if L is None:\n        import experiment_ledger as L\n    out = {}\n    for g in L.group([L.normalize(r) for r in records]):\n        if g["kernel"] and g["n"]:\n            out.setdefault(g["kernel"], []).extend(g["scores"])\n    return out\n\n\ndef pool_hidden(scores, pools, drop_zero=True):\n    """Pure. {slug: (mean, n)}. A slug in `pools` is scored over the listed slugs\' draws together (the same config\n    submitted under several slugs). A 0.00 is a failed run (server down, timeout), not a measurement of the config, so it\n    is dropped by default and the count of dropped zeros is returned. The count is of distinct failed DRAWS: a draw\n    that belongs to a pool and to its own slug is counted once (2026-10-28: it was counted per slug, so the report said\n    5 zero-score draws where Kaggle had 3)."""\n    out, used = {}, set()\n    for slug in set(scores) | set(pools):\n        members = pools.get(slug, [slug])\n        used.update(members)\n        xs = [x for s in members for x in scores.get(s, [])]\n        if drop_zero:\n            xs = [x for x in xs if x != 0]\n        if xs:\n            out[slug] = (sum(xs) / len(xs), len(xs))\n    dropped = sum(1 for s in used for x in scores.get(s, []) if x == 0) if drop_zero else 0\n    return out, dropped\n\n\ndef screen_values(traces, kernels, baselines, row_fn=None):\n    """{slug: official % mean over its mapped dirs that exist and score}; slugs with no scorable dir are left out."""\n    if row_fn is None:\n        import paper_public_eval as P\n\n        def row_fn(d):\n            rows = P.run_rows(d, baselines)\n            return P.table_row(rows)["official"] if rows else None\n    out = {}\n    for slug, dirs in kernels.items():\n        vals = []\n        for name in dirs:\n            p = os.path.join(traces, name)\n            if os.path.isdir(p):\n                v = row_fn(p)\n                if v is not None:\n                    vals.append(100.0 * v if v <= 1.5 else float(v))   # paper_public_eval reports a fraction\n        if vals:\n            out[slug] = sum(vals) / len(vals)\n    return out\n\n\ndef report(hidden, screen, base, noise=1.48):\n    """Pure. hidden {slug: (mean, n)}, screen {slug: official %} -> output lines (last line = the verdict)."""\n    lines = []\n    if base not in screen:\n        return ["SCREEN_CALIB_INSUFFICIENT base kernel %s has no scorable screen" % base]\n    joined = sorted(k for k in screen if k in hidden)\n    pending = sorted(k for k in screen if k not in hidden)\n    if base not in hidden:\n        return ["SCREEN_CALIB_INSUFFICIENT base kernel %s has no hidden score yet" % base]\n    hb, sb = hidden[base][0], screen[base]\n    lines.append("  %-38s hidden %6.2f (n=%d)  screen %5.1f   [base]" % (base, hb, hidden[base][1], sb))\n    agree = disagree = ties = 0\n    for k in sorted(joined, key=lambda k: -hidden[k][0]):\n        if k == base:\n            continue\n        dh, ds = hidden[k][0] - hb, screen[k] - sb\n        kind = "TIE" if abs(dh) < noise else ("AGREE" if dh * ds > 0 else "DISAGREE")\n        agree, disagree, ties = agree + (kind == "AGREE"), disagree + (kind == "DISAGREE"), ties + (kind == "TIE")\n        lines.append("  %-38s hidden %6.2f (n=%d)  screen %5.1f   vs base: hidden %+.2f, screen %+.1f  %s"\n                     % (k, hidden[k][0], hidden[k][1], screen[k], dh, ds, kind))\n    for k in pending:\n        lines.append("  %-38s hidden  (none usable: pending or a failed run)  screen %5.1f   prediction on record: screen %+.1f vs base"\n                     % (k, screen[k], screen[k] - sb))\n    rows = [{"kernel": k, "hidden": hidden[k][0], "offline": screen[k]} for k in joined]\n    if len(rows) < 2:\n        lines.append("SCREEN_CALIB_INSUFFICIENT %d kernel(s) have both a screen and a hidden score" % len(rows))\n        return lines\n    pa, pd, pt, _ = R.pairs(rows, noise)\n    rho = R.spearman([r["offline"] for r in rows], [r["hidden"] for r in rows]) if len(rows) >= 4 else None\n    # cmd-safe on purpose (no | & < > ^ %): the klog task passes this line to the Slack broker on a cmd command line.\n    lines.append("SCREEN_CALIB vs-base agree=%d disagree=%d ties=%d; pairs agree=%d disagree=%d ties=%d, noise %.2f; "\n                 "rho=%s n=%d" % (agree, disagree, ties, pa, pd, pt, noise, ("%.2f" % rho) if rho is not None else "n/a under 4",\n                                  len(rows)))\n    return lines\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    ap.add_argument("--from-file")\n    ap.add_argument("--map", default=MAP_FILE)\n    ap.add_argument("--traces", default=DEFAULT_TRACES)\n    ap.add_argument("--env-dir")\n    ap.add_argument("--noise", type=float, default=1.48)\n    ap.add_argument("--keep-zero", action="store_true", help="count 0.00 draws (failed runs) as scores")\n    a = ap.parse_args(argv)\n    base, kernels, pools = load_map(a.map, with_pools=True)\n    import duck_score_estimate as DS\n    env = a.env_dir or DS.ENV_DIR\n    if not os.path.isdir(env) and env.startswith("/mnt/c/"):\n        env = "C:/" + env[len("/mnt/c/"):]\n    screen = screen_values(a.traces, kernels, DS.load_baselines(env))\n    try:\n        import experiment_ledger as L\n        if a.from_file:\n            with open(a.from_file, encoding="utf-8") as f:\n                records = L.extract_records(f.read())\n        else:\n            records = L.fetch_live()\n    except Exception as ex:  # live list or file read; report, do not crash a scheduled caller\n        print("SCREEN_CALIB_ERROR %s" % str(ex)[:200])\n        return 3\n    hidden, dropped = pool_hidden(hidden_scores(records), pools, drop_zero=not a.keep_zero)\n    out = report(hidden, screen, base, a.noise)\n    if dropped:\n        out.insert(-1, "  (%d zero-score draw(s) dropped as failed runs; --keep-zero counts them)" % dropped)\n    print("\\n".join(out))\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'
open('screen_calibration.py', 'w', encoding='utf-8').write(_SRC)
print('wrote screen_calibration.py', len(_SRC), 'chars')


## Trace directories

In [ ]:
import os, sys, glob
TRACE_ROOT = os.environ.get("FI939_TRACE_ROOT", "/kaggle/input/fi939-m2-traces")
ENV_DIR = os.environ.get("FI939_ENV_DIR", "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files")
OUT = os.environ.get("FI939_OUT", "/kaggle/working" if os.path.isdir("/kaggle/working") else ".")
sys.path.insert(0, os.getcwd())
ARMS = [('base', 'duck-franzen-m2-loadsim'), ("D'", 'duck-franzen-m2-dprime-loadsim'), ('unanchor', 'duck-franzen-m2-unanchor-loadsim'), ('ctrlprobe', 'duck-franzen-m2-ctrlprobe-loadsim')]
TARGETED = [('tgt-base', 'duck-franzen-m2-tgt-base'), ('ctrlprobe2', 'duck-franzen-m2-tgt-ctrlprobe2'), ('inventory', 'duck-franzen-m2-tgt-inventory'), ('trackrecord', 'duck-franzen-m2-tgt-trackrecord')]
def present(pairs):
    got = []
    for label, d in pairs:
        p = os.path.join(TRACE_ROOT, d)
        n = len(glob.glob(os.path.join(p, "**", "*_events.jsonl"), recursive=True))
        print("%-10s %-40s %s" % (label, d, ("%d events files" % n) if n else "MISSING (skipped)"))
        if n:
            got.append((label, p))
    return got
arms = present(ARMS)
targeted = present(TARGETED)


## Power of one loadsim (paper section 5)

In [ ]:
import loadsim_power as P
if len(arms) >= 2:
    for metric in ("levels", "l2plus"):
        print(P.render(P.analyze(arms, metric, (4, 8, 16, 32))))
        print()
    if os.path.isdir(ENV_DIR):
        import duck_score_estimate as S
        r = P.analyze(arms, "score", (4, 16), S.load_baselines(ENV_DIR))
        print(P.render(r) if not r.get("error") else r)
    else:
        print("official-estimate metric skipped: no environment_files at", ENV_DIR)
else:
    print("power table needs at least two arms")


## Figure 2

In [ ]:
import make_fig2 as F
if len(arms) >= 2:
    d = F.compute(arms, 32)
    if d.get("error"):
        print("figure 2 skipped:", d)
    else:
        try:   # drawing needs matplotlib; without it only this figure is skipped (2026-10-28: the push gate's
            F.draw(d, os.path.join(OUT, "fig2_power"))   # python has none, and the error killed every later section)
            print("Figure 2 written to", os.path.join(OUT, "fig2_power.png"), "- MDE at 4 passes %.1f%%" % dict(d["curve"])[4])
        except ImportError as ex:
            print("figure 2 skipped:", ex)


## What probe_controls executed (probe audit)

In [ ]:
import probe_audit as A
cp = dict(arms).get("ctrlprobe")
if cp:
    per, n = A.audit(cp)
    print(A.render(per)[0])
else:
    print("probe audit skipped: no ctrlprobe arm")


## T4: last control family, within game (won vs failed L2+ levels)

In [ ]:
import control_mix as C
base = dict(arms).get("base")
if base:
    rows = C.analyse(C.load([base]))
    wg = C.within_game(rows)
    both = {g: v for g, v in wg.items() if v["won"] and v["failed"]}
    mean = lambda xs: sum(xs) / len(xs)
    for g in sorted(wg):
        v = wg[g]
        print("%-5s won %2d %s   failed %2d %s" % (g, len(v["won"]), ("%.2f" % mean(v["won"])) if v["won"] else "   -",
              len(v["failed"]), ("%.2f" % mean(v["failed"])) if v["failed"] else "   -"))
    if both:
        later = sum(1 for v in both.values() if mean(v["failed"]) > mean(v["won"]))
        print("WITHIN-GAME over %d games with both: mean won=%.2f failed=%.2f; failed later in %d/%d" % (
            len(both), mean([mean(v["won"]) for v in both.values()]), mean([mean(v["failed"]) for v in both.values()]),
            later, len(both)))


## Per-action effect perception: statements, tests, revision (38bu-38bx)

In [ ]:
import random, claim_null as CN, replay_tests as RT, replay_revise as RV
pair = [p for l, p in arms if l in ("base", "D'")]
if pair:
    print("## negative-control statements vs matched null (38bu)")
    print(CN.summarize(CN.run(pair, 20), random.Random(939), 1000))
    print("## announced tests: expected information and surprise (38bw)")
    r, ann, _ = RT.run(pair, 5)
    print(RT.summarize(r, ann))
    print(RT.strata(r, ann))
    print("## revision after a contradicted no-op statement (38bx)")
    print(RV.summarize(RV.run(pair, ("DURABLE", "REPORT"))))
else:
    print("effect-perception results skipped: neither base nor D' present")


## Pre-registered targeted gate (6 games x 16 passes)

In [ ]:
import targeted_gate as G
tb = dict(targeted).get("tgt-base")
for label in ("ctrlprobe2", "inventory", "trackrecord"):
    arm = dict(targeted).get(label)
    if tb and arm:
        r = G.evaluate(P.load_arm(tb, "levels"), P.load_arm(arm, "levels"), ["lf52", "bp35", "sk48"], ["r11l", "sc25", "ar25"])
        print("== %s vs tgt-base" % label)
        print(G.render(r) if not r.get("error") else r)
    else:
        print("targeted gate for %s skipped: runs not present" % label)


## HF screens (25 games x 4 passes): trace directories

In [ ]:
SCREEN_BASE = ['hf-base', 'hf-base-r2']
SCREEN_ARMS = [('freshlook', 'hf-harn-freshlook'), ('pastnotes', 'hf-harn-pastnotes'), ('triedhere', 'hf-harn-triedhere'), ('worldmodel', 'hf-worldmodel'), ('worldmodel2', 'hf-harn-wm2'), ('automacro', 'hf-harn-amacro'), ('levelpin', 'hf-harn-levelpin-r3'), ('compact32', 'hf-harn-compact32-r2'), ('deep140', 'hf-harn-deep140'), ('aggr1', 'hf-harn-aggr1'), ('effort-medium', 'hf-effort-medium'), ('effort-low', 'hf-effort-low'), ('KTO-G5', 'hf-lora-g5'), ('KTO-G12', 'hf-lora-g12'), ('KTO-G12r', 'hf-lora-g12r')]
HELDOUT = ['ar25', 'bp35', 'cd82', 'dc22', 'g50t', 'ka59', 'lp85', 'r11l', 're86', 'sb26', 'sk48', 'su15', 'tu93', 'vc33']
def screen_dir(name):
    return os.path.join(TRACE_ROOT, name)
def screen_present(pairs):
    got = []
    for label, d in pairs:
        n = len(glob.glob(os.path.join(screen_dir(d), "**", "*_events.jsonl"), recursive=True))
        print("%-14s %-26s %s" % (label, d, ("%d pass logs" % n) if n else "MISSING screen (skipped)"))
        if n:
            got.append((label, d))
    return got
base_screens = screen_present([("base", d) for d in SCREEN_BASE])
screens = screen_present(SCREEN_ARMS)


## Offline screen gate for every arm (paper Results table)

In [ ]:
import fi939_gate as FG
if len(base_screens) == len(SCREEN_BASE):
    base_pg = FG.pooled_per_game(TRACE_ROOT, ",".join(SCREEN_BASE))
    for label, d in screens:
        r = FG.evaluate(base_pg, FG.per_game(screen_dir(d)), HELDOUT, min_heldout_gain_pct=5.0)
        if r["verdict"] == "CANNOT_EVALUATE":
            print("%-14s cannot evaluate (%d shared games, missing held-out %s)" % (label, r["games"], r["missing_heldout"]))
        else:
            print("%-14s held-out %+6.1f%%  all levels %+6.1f%%  p_worse %.3f  %s" % (
                label, r["heldout_pct"], r["total_pct"], r["p_worse"], r["verdict"]))
else:
    print("screen gate skipped: base screens missing")


## Public evaluation set, official metric

In [ ]:
import paper_public_eval as PE, duck_score_estimate as S
if os.path.isdir(ENV_DIR):
    bl = S.load_baselines(ENV_DIR)
    res = []
    for label, d in [("base", SCREEN_BASE[0]), ("base, repeat", SCREEN_BASE[1])] + screens:
        rows = PE.run_rows(screen_dir(d), bl) if os.path.isdir(screen_dir(d)) else []
        if rows:
            res.append((label, PE.table_row(rows)))
        else:
            print("%s: no scorable passes" % label)
    print(PE.render(res))
else:
    print("public-set official metric skipped: no environment_files at", ENV_DIR)


## Does the offline screen predict the hidden score? (screen-vs-hidden calibration)

In [ ]:
import screen_calibration as SC
HIDDEN_AS_OF = '2026-10-28'
HIDDEN_SCORES = {'duck-franzen-m2-serverguard-lb': [25.54, 25.51, 26.59, 26.43, 26.93, 27.16, 29.1, 26.65, 27.01, 24.47, 0.0, 0.0], 'duck-franzen-m2-loadguard-lb': [26.04], 'duck-franzen-m2-levelpin-lb': [28.46], 'duck-franzen-m2-lora-g5-lb': [24.12], 'duck-franzen-m2-aggr1-lb': [18.03], 'duck-franzen-m2-deep140-lb': [19.99], 'duck-franzen-m2-freshlook-lb': [0.0]}
CAL_MAP = {'base': 'duck-franzen-m2-loadguard-lb', 'kernels': {'duck-franzen-m2-loadguard-lb': ['hf-base', 'hf-base-r2'], 'duck-franzen-m2-levelpin-lb': ['hf-harn-levelpin-r3'], 'duck-franzen-m2-freshlook-lb': ['hf-harn-freshlook'], 'duck-franzen-m2-aggr1-lb': ['hf-harn-aggr1'], 'duck-franzen-m2-lora-g5-lb': ['hf-lora-g5'], 'duck-franzen-m2-deep140-lb': ['hf-harn-deep140']}, 'hidden_pool': {'duck-franzen-m2-loadguard-lb': ['duck-franzen-m2', 'duck-franzen-m2-serverguard-lb', 'duck-franzen-m2-loadguard-lb']}}
if os.path.isdir(ENV_DIR):
    hidden, dropped = SC.pool_hidden(HIDDEN_SCORES, CAL_MAP["hidden_pool"])
    screen = SC.screen_values(TRACE_ROOT, CAL_MAP["kernels"], S.load_baselines(ENV_DIR))
    print("hidden draws as of %s; failed 0.00 draws dropped: %d" % (HIDDEN_AS_OF, dropped))
    print("\n".join(SC.report(hidden, screen, CAL_MAP["base"])))
else:
    print("screen-vs-hidden calibration skipped: no environment_files at", ENV_DIR)
